# CONUS Visualizations — SIF × Irrigation × Drought

This notebook assembles all primary CONUS-scale visualizations for the
SIF × HumanET × Drought analysis. It loads pre-computed results from:

- **`01_human_et_conus.ipynb`** → processed OpenET and NLDAS ET files used to
  reconstruct `delta_maps` (HumanET = OpenET − NLDAS Noah ET)
- **`02_irrigation_sif_regression_conus.ipynb`** → `df_combined_gs.parquet`,
  `crop_mask_static.npy`, and pixel-level regression slope results
  (recomputed here if not saved to disk)

**Prerequisites:** Run notebooks `01_` and `02_` first to generate the
parquet and mask files in `data/processed/conus/regression/`.

In [1]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from scipy import stats
import rasterio
import xarray as xr

_root_env    = os.environ.get('SIF_ROOT')
project_root = Path(_root_env) if _root_env else Path('../../..').resolve()

proc = project_root / 'data' / 'processed' / 'conus'
figs = project_root / 'figures' / 'conus'
figs.mkdir(parents=True, exist_ok=True)

openet_proc = proc / 'openet'
nldas_proc  = proc / 'nldas'

print('Project root:', project_root)
print('Data dir:    ', proc)
print('Figures dir: ', figs)

Project root: /home/pielab-sandbox-jcoldiron/SIF-Analysis
Data dir:     /home/pielab-sandbox-jcoldiron/SIF-Analysis/data/processed/conus
Figures dir:  /home/pielab-sandbox-jcoldiron/SIF-Analysis/figures/conus


In [2]:
# ── ltc color palettes ────────────────────────────────────────────────────
# R package: https://github.com/loukesio/ltc-color-palettes
# Hex codes extracted from R/ltc_functions.R in the CRAN source tarball.
# No R runtime dependency — colors are hardcoded here as Python lists.
#
# ploen    — 5-color sequential (cool blue → warm brownish red)
#            Use for: data running 0 and up (mean HumanET map, Fig 3 lines)
# heatmap0 — 9-color diverging (dark teal → yellow → dark red)
#            Use for: diverging data (trend map, per-pixel slope maps)
from matplotlib.colors import LinearSegmentedColormap

PALETTE_PLOEN = ['#3F5671', '#83A1C3', '#CEB5C8', '#FAC898', '#B17776']
PALETTE_HEATMAP0 = [
    '#001219', '#005F73', '#0A9396', '#94D2BD', '#E9D8A6',
    '#EE9B00', '#CA6702', '#AE2012', '#9B2226',
]

cmap_ploen    = LinearSegmentedColormap.from_list('ploen',    PALETTE_PLOEN,    N=256)
cmap_heatmap0 = LinearSegmentedColormap.from_list('heatmap0', PALETTE_HEATMAP0, N=256)

# NaN-transparent versions for map overlays (basemap shows through missing pixels)
cmap_ploen_map    = cmap_ploen.copy();    cmap_ploen_map.set_bad('white', alpha=0)
cmap_heatmap0_map = cmap_heatmap0.copy(); cmap_heatmap0_map.set_bad('white', alpha=0)
# Red→White→Blue diverging map (negative=red, positive=blue)
# Used for trend maps and slope maps
cmap_rdbu_map = plt.cm.RdBu.copy(); cmap_rdbu_map.set_bad('white', alpha=0)
# Reversed ploen for Human ET mean (low=warm, high=blue)
cmap_ploen_r_map = cmap_ploen_map.reversed()

# Figure 3 drought-severity line colors — sampled from ploen at 4 positions
DROUGHT_COLORS_LTC = {
    'No Drought\n(SPEI>-0.5)':          PALETTE_PLOEN[0],  # '#3F5671' dark blue
    'Mild\n(-1.0<SPEI<=-0.5)':          PALETTE_PLOEN[1],  # '#83A1C3' light blue
    'Moderate\n(-1.5<SPEI<=-1.0)':      PALETTE_PLOEN[3],  # '#FAC898' warm peach
    'Severe\n(SPEI<=-1.5)':             PALETTE_PLOEN[4],  # '#B17776' brownish red
}

print('ltc palettes loaded:')
print('  ploen   :', PALETTE_PLOEN)
print('  heatmap0:', PALETTE_HEATMAP0)


ltc palettes loaded:
  ploen   : ['#3F5671', '#83A1C3', '#CEB5C8', '#FAC898', '#B17776']
  heatmap0: ['#001219', '#005F73', '#0A9396', '#94D2BD', '#E9D8A6', '#EE9B00', '#CA6702', '#AE2012', '#9B2226']


## 1. Setup and Data Loading

We reconstruct the key data objects needed for visualization:

1. **Config constants** — identical to those in `01_` and `02_` so all grids align
2. **`df_combined_gs`** — the pixel-month panel dataset (parquet)
3. **`crop_mask_static`** — static 189 × 325 boolean cropland mask (numpy)
4. **`delta_maps`** — reconstructed from the processed OpenET and NLDAS ET files
5. **Pixel regression slopes** — loaded from cache or recomputed from `df_combined_gs`

### 1.1 Configuration

In [3]:
# ── Grid and analysis constants (must match 01_ and 02_) ──────────────────
CONUS_LON = np.arange(-124.6875, -84.0625,  0.125)   # 325 longitudes
CONUS_LAT = np.arange(  49.3125,  25.6875, -0.125)   # 189 latitudes (N→S)
n_lat, n_lon = len(CONUS_LAT), len(CONUS_LON)

YEARS          = list(range(2015, 2025))   # 10-year study period
GROWING_SEASON = [4, 5, 6, 7, 8, 9]       # April–September

IRR_THRESHOLD_MM   = 20.0   # mm/month — HumanET > 20 = irrigation
DROUGHT_SPEI_THRESH = -0.5  # SPEI ≤ -0.5 = at least mild drought
MIN_OBS_PIXEL       = 5     # minimum drought-month obs per pixel for regression
ALPHA_SPATIAL       = 0.10  # significance threshold for pixel slopes

print('CONUS grid:', n_lon, 'x', n_lat, 'at 0.125 deg')
print('Study period:', YEARS[0], '-', YEARS[-1])
print('Growing season months:', GROWING_SEASON)

CONUS grid: 325 x 189 at 0.125 deg
Study period: 2015 - 2024
Growing season months: [4, 5, 6, 7, 8, 9]


### 1.2 Load Panel Dataset and Cropland Mask

In [4]:
parquet_path = proc / 'regression' / 'df_combined_gs.parquet'
mask_path    = proc / 'regression' / 'crop_mask_static.npy'

if not parquet_path.exists():
    raise FileNotFoundError(
        'Run 01_human_et_conus.ipynb first: ' + str(parquet_path)
    )

df = pd.read_parquet(parquet_path)
if 'date' not in df.columns:
    df['date'] = pd.to_datetime(df['yyyymm'], format='%Y%m')
if 'dm_cat_int' not in df.columns and 'dm_cat' in df.columns:
    df['dm_cat_int'] = df['dm_cat'].round().astype('Int64')

crop_mask_static = np.load(mask_path)

print('Loaded:', parquet_path.name)
print('Shape:', df.shape)
print('Date range:', df['date'].min().date(), 'to', df['date'].max().date())
print('Cropland pixels (mask):', crop_mask_static.sum())

Loaded: df_combined_gs.parquet
Shape: (1509180, 16)
Date range: 2015-04-01 to 2024-09-01
Cropland pixels (mask): 25153


### 1.3 Reconstruct delta_maps (HumanET = OpenET − NLDAS ET)

We reload the processed OpenET and NLDAS ET files for **growing-season months only**
(April–September, 2015–2024) and recompute HumanET = OpenET − NLDAS ET. Files are
already aligned to the 189 × 325 CONUS grid, so no reprojection is needed.
We use `crop_mask_static` for masking (same ≥ 50% cropland threshold as notebook 01).

In [5]:
delta_maps = {}   # key: (year, month) → 2D array (189, 325)
n_proc = 0
n_miss = 0

for year in YEARS:
    for month in GROWING_SEASON:
        yyyymm = str(year) + str(month).zfill(2)
        fp_openet = openet_proc / ('OpenET_CONUS_' + yyyymm + '.tif')
        fp_nldas  = nldas_proc  / ('NLDAS_Evap_'   + yyyymm + '.nc')

        openet_arr = None
        nldas_arr  = None

        if fp_openet.exists():
            try:
                with rasterio.open(fp_openet) as src:
                    a  = src.read(1).astype(float)
                    nd = src.nodata
                    if nd is not None:
                        a[a == nd] = np.nan
                    if a.shape == (n_lat, n_lon):
                        openet_arr = a
            except Exception as e:
                print('  OpenET error ' + yyyymm + ': ' + str(e))

        if fp_nldas.exists():
            try:
                ds = xr.open_dataset(fp_nldas)
                et_var = None
                for vname in ['Evap', 'EVP', 'et', 'ET']:
                    if vname in ds:
                        et_var = vname
                        break
                if et_var is None:
                    et_var = list(ds.data_vars)[0]
                arr_n = ds[et_var].values
                ds.close()
                if arr_n.ndim == 3:
                    arr_n = arr_n[0]
                if arr_n.shape == (n_lat, n_lon):
                    nldas_arr = arr_n.astype(float)
            except Exception as e:
                print('  NLDAS error ' + yyyymm + ': ' + str(e))

        if openet_arr is not None and nldas_arr is not None:
            delta = openet_arr - nldas_arr
            delta = np.where(crop_mask_static, delta, np.nan)
            delta = np.where(np.abs(delta) > 500, np.nan, delta)
            delta_maps[(year, month)] = delta
            n_proc += 1
        else:
            n_miss += 1

print('Growing-season months loaded: ' + str(n_proc))
print('Missing months:               ' + str(n_miss))
print('delta_maps entries:           ' + str(len(delta_maps)))

Growing-season months loaded: 60
Missing months:               0
delta_maps entries:           60


### 1.4 Pixel-Level Regression Slopes

We check for a cached results file (`pix_slope_results.csv`) written by notebook 02.
If not found we recompute using the identical logic: OLS of `sif_z ~ delta_et`
restricted to drought months (SPEI ≤ −0.5), one regression per cropland pixel,
minimum 5 observations, retaining results at p < 0.10.

In [6]:
_slope_cache = proc / 'regression' / 'pix_slope_results.csv'

if _slope_cache.exists():
    pix_stats = pd.read_csv(_slope_cache)
    print('Loaded pixel slopes from cache:', _slope_cache.name)
else:
    print('Cache not found — recomputing pixel-level slopes...')

    # Build complete-case regression dataset (same as notebook 02)
    df_reg = df.dropna(subset=['sif_z', 'spei90d', 'delta_et']).copy()
    df_reg = df_reg[
        np.isfinite(df_reg['sif_z']) &
        np.isfinite(df_reg['spei90d']) &
        np.isfinite(df_reg['delta_et'])
    ].copy()

    # Filter to drought months only
    df_drought = df_reg[df_reg['spei90d'] <= DROUGHT_SPEI_THRESH].copy()
    print('Drought-month obs:', len(df_drought))

    def _pixel_slope(grp):
        xy = grp[['delta_et', 'sif_z']].dropna()
        if len(xy) < MIN_OBS_PIXEL:
            return pd.Series({'slope': np.nan, 'pval': np.nan, 'n': len(xy)})
        slope, _, _, pval, _ = stats.linregress(
            xy['delta_et'].values, xy['sif_z'].values
        )
        return pd.Series({'slope': slope, 'pval': pval, 'n': len(xy)})

    print('Running pixel regressions (~30-60 s)...')
    pix_stats = (
        df_drought
        .groupby(['lat', 'lon'])
        .apply(_pixel_slope)
        .reset_index()
    )
    print('Pixel regression complete.')

# Significant pixels at p < ALPHA_SPATIAL
pix_sig = pix_stats[pix_stats['pval'] < ALPHA_SPATIAL].copy()
n_pos   = int((pix_sig['slope'] > 0).sum())
n_neg   = int((pix_sig['slope'] < 0).sum())

print()
print('Pixels with >= ' + str(MIN_OBS_PIXEL) + ' drought obs: '
      + str(pix_stats['slope'].notna().sum()))
print('Significant at p < ' + str(ALPHA_SPATIAL) + ': ' + str(len(pix_sig)))
print('  Positive slope (buffering): ' + str(n_pos))
print('  Negative slope:             ' + str(n_neg))

Cache not found — recomputing pixel-level slopes...
Drought-month obs: 120766
Running pixel regressions (~30-60 s)...


Pixel regression complete.

Pixels with >= 5 drought obs: 7456
Significant at p < 0.1: 2032
  Positive slope (buffering): 1701
  Negative slope:             331


---

## 2. Human ET Change Over Time Map

This map shows **where HumanET (OpenET − NLDAS ET) has been increasing or decreasing
over the 2015–2024 study period** across CONUS cropland.

**Method:**
1. For each year, compute the mean of the six growing-season monthly HumanET values
   at each cropland pixel, giving a single annual estimate (mm/month).
2. Fit a linear trend (OLS) across the 10 annual means using `scipy.stats.linregress`.
3. Report the slope in **mm/month per year** — positive = HumanET increasing,
   negative = decreasing.

**Interpretation:** A positive trend may reflect expanded irrigation, improved
crop water use, or wetter conditions driving higher OpenET. A negative trend can
indicate crop abandonment, drought-limited production, or switching to less
water-intensive crops.

### 2.1 Compute Per-Pixel Trend Slopes

In [7]:
# ── Per-year mean growing-season HumanET at each pixel ───────────────────
yearly_gs = {}
for year in YEARS:
    arrs = [delta_maps[(year, m)] for m in GROWING_SEASON
            if (year, m) in delta_maps]
    if arrs:
        yearly_gs[year] = np.nanmean(np.stack(arrs, axis=0), axis=0)
    else:
        yearly_gs[year] = np.full((n_lat, n_lon), np.nan)

# Stack: shape (n_years, n_lat, n_lon)
gs_stack  = np.stack([yearly_gs[y] for y in YEARS], axis=0)
year_vals = np.array(YEARS, dtype=float)
print('Yearly GS mean stack shape:', gs_stack.shape)

# ── Linear trend at each cropland pixel ───────────────────────────────────
trend_slope = np.full((n_lat, n_lon), np.nan)
trend_pval  = np.full((n_lat, n_lon), np.nan)

crop_rows, crop_cols = np.where(crop_mask_static)
for ri, ci in zip(crop_rows, crop_cols):
    y     = gs_stack[:, ri, ci]
    valid = np.isfinite(y)
    if valid.sum() < 3:
        continue
    slope, _, _, pval, _ = stats.linregress(year_vals[valid], y[valid])
    trend_slope[ri, ci] = slope
    trend_pval[ri, ci]  = pval

# ── Summary statistics ─────────────────────────────────────────────────────
finite_slopes = trend_slope[np.isfinite(trend_slope)]
mean_trend    = float(np.nanmean(finite_slopes))
frac_inc      = float((finite_slopes > 0).mean())
frac_dec      = 1.0 - frac_inc

print('Trend computed for ' + str(len(finite_slopes)) + ' cropland pixels')
print('Slope range: ' + str(round(float(finite_slopes.min()), 3))
      + ' to ' + str(round(float(finite_slopes.max()), 3)) + ' mm/month/year')
print('CONUS mean trend:  ' + str(round(mean_trend, 4)) + ' mm/month/year')
print('Pixels increasing: ' + str(round(100 * frac_inc, 1)) + '%')
print('Pixels decreasing: ' + str(round(100 * frac_dec, 1)) + '%')

Yearly GS mean stack shape: (10, 189, 325)


/tmp/ipykernel_2204395/1085650207.py:7: RuntimeWarning: Mean of empty slice
  yearly_gs[year] = np.nanmean(np.stack(arrs, axis=0), axis=0)


Trend computed for 7595 cropland pixels
Slope range: -20.574 to 11.72 mm/month/year
CONUS mean trend:  0.8313 mm/month/year
Pixels increasing: 80.0%
Pixels decreasing: 20.0%


### 2.2 CONUS Trend Map

In [8]:
# ── NOTE: this standalone trend map is kept for quick reference.
# The publication-quality two-panel figure is in Section 4 below.
# Requires Section 4.1 helper cells (_X5070, _Y5070, _draw_states,
# _add_basemap, _albers_axes). Run Section 4.1 first if standalone.
# ─────────────────────────────────────────────────────────────────────────

try:
    _X5070
except NameError:
    from pyproj import Transformer as _Tr
    _t5070 = _Tr.from_crs('EPSG:4326', 'EPSG:5070', always_xy=True)
    _LON_GRID, _LAT_GRID = np.meshgrid(CONUS_LON, CONUS_LAT)
    _X5070, _Y5070 = _t5070.transform(_LON_GRID, _LAT_GRID)
    _xmin, _xmax = float(_X5070.min()) - 80_000, float(_X5070.max()) + 80_000
    _ymin, _ymax = float(_Y5070.min()) - 80_000, float(_Y5070.max()) + 80_000

# ── Color scale ───────────────────────────────────────────────────────────
_fs = trend_slope[np.isfinite(trend_slope)]
_vmax = max(float(np.nanpercentile(np.abs(_fs), 97)), 0.10)

_trend_masked = np.where(crop_mask_static, trend_slope, np.nan)

_cmap_t = plt.cm.RdBu.copy()
_cmap_t.set_bad('white', alpha=0)

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4.5))

im = ax.pcolormesh(
    _X5070, _Y5070, _trend_masked,
    cmap=_cmap_t, vmin=-_vmax, vmax=_vmax,
    shading='nearest', rasterized=True, zorder=2,
)

# Apply axes formatting (inline fallback if Section 4.1 not yet run)
try:
    _albers_axes(ax)
except NameError:
    ax.set_xticks([]); ax.set_yticks([])
    for _sp in ax.spines.values(): _sp.set_visible(False)
    ax.set_aspect('equal')
    ax.set_xlim(_xmin, _xmax)
    ax.set_ylim(_ymin, _ymax)

try:
    _add_basemap(ax)
except NameError:
    pass

try:
    _draw_states(ax, lw=0.4, color='#333333')
except NameError:
    pass

cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, shrink=0.85)
cb.set_label('HumanET linear trend (mm/month/year)  |  Blue = increasing  /  Red = decreasing',
             fontsize=9)

_title_str = (
    'Linear Trend in Growing-Season HumanET (OpenET - NLDAS ET) 2015-2024 - CONUS Cropland'
)
_sub_str = (
    'CONUS mean: {:.3f} mm/month/yr  |  Increasing: {:.1f}%  |  Decreasing: {:.1f}%'.format(
        mean_trend, 100 * frac_inc, 100 * frac_dec
    )
)
ax.set_title(_title_str + '\n' + _sub_str, fontsize=10)

plt.tight_layout()
plt.savefig(figs / 'human_et_trend_map.png', dpi=300, bbox_inches='tight')
plt.savefig(figs / 'human_et_trend_map.pdf', bbox_inches='tight')
plt.show()
print('Saved: human_et_trend_map.png / .pdf  (Albers EPSG:5070, 300 DPI)')


Saved: human_et_trend_map.png / .pdf  (Albers EPSG:5070, 300 DPI)


---

## 3. Positive vs. Negative Pixel Regression Slopes

This map shows **where irrigation is statistically associated with higher SIF
z-scores during drought months** across CONUS cropland.

Each pixel represents an independent OLS regression of SIF anomaly (`sif_z`) on
HumanET (`delta_et`), estimated using drought-month observations only
(SPEI-90d ≤ −0.5, ≥ 5 observations per pixel). Significance threshold: p < 0.10.

**Color encoding:**
- **Blue** — positive slope: more HumanET → higher SIF z-score during drought
  (irrigation buffers crop photosynthesis under water stress)
- **Red** — negative slope: more HumanET → lower SIF z-score
  (no buffering or counter-intuitive signal, e.g. stressed irrigated fields
  in areas with severe groundwater depletion)
- **Light gray** — non-significant cropland pixels (p ≥ 0.10 or < 5 drought obs)

This is a descriptive spatial analysis; causal interpretation requires the
pooled regression in notebook 02.

In [9]:
# ── Build lat/lon → row/col lookup for fast grid placement ───────────────
_lat_to_row = {round(float(v), 4): i for i, v in enumerate(CONUS_LAT)}
_lon_to_col = {round(float(v), 4): i for i, v in enumerate(CONUS_LON)}

# ── Categorical map: NaN=outside cropland, 0=non-sig, +1=pos, -1=neg ─────
cat_map = np.full((n_lat, n_lon), np.nan)
cat_map[crop_mask_static] = 0.0      # all cropland pixels start as non-sig

n_placed = 0
for _, row in pix_sig.iterrows():
    ri = _lat_to_row.get(round(float(row['lat']), 4))
    ci = _lon_to_col.get(round(float(row['lon']), 4))
    if ri is not None and ci is not None:
        cat_map[ri, ci] = 1.0 if row['slope'] > 0 else -1.0
        n_placed += 1

print('Significant pixels placed on grid: ' + str(n_placed) + ' / ' + str(len(pix_sig)))
print('  Positive (blue): ' + str(n_pos))
print('  Negative (red):  ' + str(n_neg))
print('  Non-sig (gray):  '
      + str(int((cat_map == 0).sum())))

# ── Custom 3-class colormap ────────────────────────────────────────────────
# Values: -1 → red, 0 → light gray, +1 → blue
_cmap   = ListedColormap(['#C62828', '#CCCCCC', '#1565C0'])  # red, gray, blue
_bounds = [-1.5, -0.5, 0.5, 1.5]
_norm   = BoundaryNorm(_bounds, _cmap.N)

fig, ax = plt.subplots(figsize=(17, 7))

ax.imshow(
    cat_map,
    extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
    origin='upper',
    cmap=_cmap,
    norm=_norm,
    aspect='auto',
)

# ── Legend ─────────────────────────────────────────────────────────────────
legend_elements = [
    Patch(facecolor='#1565C0', edgecolor='none',
          label='Positive slope — irrigation buffers SIF (n=' + str(n_pos) + ')'),
    Patch(facecolor='#C62828', edgecolor='none',
          label='Negative slope — no buffering benefit (n=' + str(n_neg) + ')'),
    Patch(facecolor='#CCCCCC', edgecolor='none',
          label='Non-significant cropland (p >= ' + str(ALPHA_SPATIAL)
                + ' or < ' + str(MIN_OBS_PIXEL) + ' obs)'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=10, framealpha=0.9)

ax.set_title(
    'Pixel-Level Irrigation Buffering of SIF Under Drought - CONUS Cropland\n'
    'OLS: SIF z-score ~ HumanET | Drought months (SPEI <= '
    + str(DROUGHT_SPEI_THRESH)
    + ')  |  Significant at p < '
    + str(ALPHA_SPATIAL),
    fontsize=12
)
ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude',  fontsize=11)
ax.grid(alpha=0.15)

plt.tight_layout()
plt.savefig(figs / 'reg_sif_slope_positive_negative.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reg_sif_slope_positive_negative.png')

Significant pixels placed on grid: 2032 / 2032
  Positive (blue): 1701
  Negative (red):  331
  Non-sig (gray):  23121


Saved: reg_sif_slope_positive_negative.png


---

## 3.5 Diagnosis — Border Artifacts, Aspect Ratio, and State Boundaries

Before producing the publication figure we diagnose the three known problems in the
earlier draft maps:

1. **Border artifacts** — spurious NLDAS pixels appearing outside the US political boundary
   (especially southern AZ/NM/TX and northern WA/ID). A pixel can pass the 50% cropland CDL
   threshold even if only part of it falls inside the US, because CDL is masked by land cover,
   not by political boundary.

2. **Squished aspect ratio** — `imshow` with `aspect='auto'` in geographic coordinates
   (EPSG:4326) renders 1° lon ≡ 1° lat, which is incorrect at mid-latitudes.

3. **Missing state boundaries** — no political reference overlay.

### Diagnostic cells:
- **(a)** CDL cropland mask extent vs. US states dissolved boundary
- **(b)** CRS / transform / shape of each input dataset
- **(c)** Spot-check values in border-artifact regions
- **(d)** Diagnosis summary

In [10]:
# ── Diagnostic (a): CDL mask extent vs. US states dissolved boundary ─────
import geopandas as gpd

_states_fp = (project_root / 'data' / 'processed' / 'conus' / 'us_states_simple.geojson').resolve()

if _states_fp.exists():
    _states_gdf  = gpd.read_file(_states_fp)
    _us_dissolved = _states_gdf.dissolve()     # single polygon = full CONUS

    # Bounding box of US dissolved polygon
    _us_bounds = _us_dissolved.total_bounds   # (minx, miny, maxx, maxy)
    print('US boundary extent (EPSG:4326):')
    print('  lon: {:.4f} to {:.4f}'.format(_us_bounds[0], _us_bounds[2]))
    print('  lat: {:.4f} to {:.4f}'.format(_us_bounds[1], _us_bounds[3]))
else:
    print('WARNING: us_states_simple.geojson not found — skipping boundary diagnostics')
    _us_dissolved = None

# Bounding box of CONUS grid (all pixels)
print('\nCONUS NLDAS grid extent (EPSG:4326):')
print('  lon: {:.4f} to {:.4f}'.format(float(CONUS_LON[0] - 0.0625), float(CONUS_LON[-1] + 0.0625)))
print('  lat: {:.4f} to {:.4f}'.format(float(CONUS_LAT[-1] - 0.0625), float(CONUS_LAT[0]  + 0.0625)))

# Cropland mask pixels
_cmask_rows, _cmask_cols = np.where(crop_mask_static)
_cmask_lats = CONUS_LAT[_cmask_rows]
_cmask_lons = CONUS_LON[_cmask_cols]
print('\nCropland mask pixel extent (EPSG:4326):')
print('  lon: {:.4f} to {:.4f}'.format(float(_cmask_lons.min()), float(_cmask_lons.max())))
print('  lat: {:.4f} to {:.4f}'.format(float(_cmask_lats.min()), float(_cmask_lats.max())))

# Check for cropland pixels outside the US boundary
if _us_dissolved is not None:
    from shapely.geometry import box as _box, MultiPoint as _MP
    # Sample every 5th cropland pixel to keep it fast
    _sample_pts = gpd.GeoDataFrame(
        {'geometry': [_box(lo - 0.0625, la - 0.0625, lo + 0.0625, la + 0.0625)
                      for la, lo in zip(_cmask_lats[::5], _cmask_lons[::5])]},
        crs='EPSG:4326',
    )
    _inside = _sample_pts.within(_us_dissolved.geometry.iloc[0])
    n_out = int((~_inside).sum())
    n_tot = int(len(_inside))
    print('\n(a) {}/{} sampled CDL cropland pixels (every 5th) fall OUTSIDE US boundary'.format(
        n_out, n_tot
    ))
    if n_out > 0:
        print('    => Border artifacts confirmed: CDL mask alone is insufficient.')
    else:
        print('    => No border artifacts detected in sample.')

US boundary extent (EPSG:4326):
  lon: -124.7000 to -67.0000
  lat: 24.5000 to 49.0000

CONUS NLDAS grid extent (EPSG:4326):
  lon: -124.7500 to -84.1250
  lat: 25.7500 to 49.3750

Cropland mask pixel extent (EPSG:4326):
  lon: -124.6875 to -84.1875
  lat: 25.8125 to 49.3125

(a) 3391/5031 sampled CDL cropland pixels (every 5th) fall OUTSIDE US boundary
    => Border artifacts confirmed: CDL mask alone is insufficient.


In [11]:
# ── Diagnostic (b): CRS, transform, and shape of each input dataset ──────
print('(b) Dataset grid/CRS summary')
print()

# CONUS NLDAS grid (inferred from array constants)
print('NLDAS CONUS analysis grid:')
print('  Shape : {} lat x {} lon'.format(n_lat, n_lon))
print('  Res   : 0.125 deg')
print('  CRS   : EPSG:4326 (geographic, WGS84)')
print('  Extent: lon {:.4f}–{:.4f}, lat {:.4f}–{:.4f}'.format(
    float(CONUS_LON[0]), float(CONUS_LON[-1]),
    float(CONUS_LAT[-1]), float(CONUS_LAT[0])
))
print()

# Spot-check one OpenET GeoTIFF
_oe_sample = next(iter(sorted((openet_proc).glob('OpenET_CONUS_*.tif'))), None)
if _oe_sample:
    with rasterio.open(_oe_sample) as _src:
        print('OpenET GeoTIFF ({}):'.format(_oe_sample.name))
        print('  Shape    :', _src.height, 'x', _src.width)
        print('  CRS      :', _src.crs)
        print('  Transform:', _src.transform)
        print('  NoData   :', _src.nodata)
        print()

# Spot-check one NLDAS Noah NetCDF
_nldas_sample = next(iter(sorted((nldas_proc).glob('NLDAS_Evap_*.nc'))), None)
if _nldas_sample:
    _ds = xr.open_dataset(_nldas_sample)
    _lat_arr = _ds.coords.get('lat', _ds.coords.get('latitude', None))
    _lon_arr = _ds.coords.get('lon', _ds.coords.get('longitude', None))
    print('NLDAS Noah NetCDF ({}):'.format(_nldas_sample.name))
    print('  Variables:', list(_ds.data_vars)[:6])
    if _lat_arr is not None:
        print('  Lat range : {:.4f} to {:.4f}'.format(float(_lat_arr.min()), float(_lat_arr.max())))
    if _lon_arr is not None:
        print('  Lon range : {:.4f} to {:.4f}'.format(float(_lon_arr.min()), float(_lon_arr.max())))
    _ds.close()
    print()

# US states shapefile
if _states_fp.exists():
    print('US States GeoJSON:')
    print('  CRS      :', _states_gdf.crs)
    print('  N rows   :', len(_states_gdf))
    _sb = _states_gdf.total_bounds
    print('  Bounds   : lon {:.2f}–{:.2f}, lat {:.2f}–{:.2f}'.format(_sb[0], _sb[2], _sb[1], _sb[3]))
print()
print('=> All datasets should be EPSG:4326. Pixel centres align at 0.125° steps.')

(b) Dataset grid/CRS summary

NLDAS CONUS analysis grid:
  Shape : 189 lat x 325 lon
  Res   : 0.125 deg
  CRS   : EPSG:4326 (geographic, WGS84)
  Extent: lon -124.6875–-84.1875, lat 25.8125–49.3125

OpenET GeoTIFF (OpenET_CONUS_201501.tif):
  Shape    : 189 x 325
  CRS      : EPSG:4326
  Transform: | 0.12, 0.00,-124.75|
| 0.00,-0.12, 49.38|
| 0.00, 0.00, 1.00|
  NoData   : nan

NLDAS Noah NetCDF (NLDAS_Evap_201501.nc):
  Variables: ['Evap']
  Lat range : 25.8125 to 49.3125
  Lon range : -124.6875 to -84.1875

US States GeoJSON:
  CRS      : EPSG:4326
  N rows   : 48
  Bounds   : lon -124.70–-67.00, lat 24.50–49.00

=> All datasets should be EPSG:4326. Pixel centres align at 0.125° steps.


In [12]:
# ── Diagnostic (c): Spot-check values in suspected border-artifact regions ─
# Focus on southern AZ/NM/TX (~lat 31–33 N) and northern WA/ID (~lat 48–49 N)
print('(c) Spot-check HumanET values in border-artifact regions')
print()

_region_checks = [
    ('Southern AZ/NM/TX',     (31.0, 33.0), (-115.0, -103.0)),
    ('Northern WA/ID border', (48.0, 49.5), (-124.0, (-117.0))),
]

_all_delta = np.nanmean(np.stack(list(delta_maps.values()), axis=0), axis=0)

for label, (lat_lo, lat_hi), (lon_lo, lon_hi) in _region_checks:
    _row_lo = int(np.argmin(np.abs(CONUS_LAT - lat_hi)))   # hi lat = low row index (N→S)
    _row_hi = int(np.argmin(np.abs(CONUS_LAT - lat_lo)))
    _col_lo = int(np.argmin(np.abs(CONUS_LON - lon_lo)))
    _col_hi = int(np.argmin(np.abs(CONUS_LON - lon_hi)))

    _patch = _all_delta[_row_lo:_row_hi+1, _col_lo:_col_hi+1]
    _mask_patch = crop_mask_static[_row_lo:_row_hi+1, _col_lo:_col_hi+1]
    _nonnan = _patch[np.isfinite(_patch)]
    _masked_nonnan = _patch[_mask_patch & np.isfinite(_patch)]

    print('{} (lat {:.0f}–{:.0f}N, lon {:.0f}–{:.0f}W):'.format(
        label, lat_lo, lat_hi, -lon_hi, -lon_lo))
    print('  Patch shape        : {}×{}'.format(_patch.shape[0], _patch.shape[1]))
    print('  All non-NaN vals   : {}  range: {:.1f} to {:.1f} mm/mo'.format(
        len(_nonnan), float(_nonnan.min()) if len(_nonnan) else np.nan,
        float(_nonnan.max()) if len(_nonnan) else np.nan))
    print('  CDL-masked non-NaN : {}  range: {:.1f} to {:.1f} mm/mo'.format(
        len(_masked_nonnan),
        float(_masked_nonnan.min()) if len(_masked_nonnan) else np.nan,
        float(_masked_nonnan.max()) if len(_masked_nonnan) else np.nan))
    print()

(c) Spot-check HumanET values in border-artifact regions

Southern AZ/NM/TX (lat 31–33N, lon 103–115W):
  Patch shape        : 17×97
  All non-NaN vals   : 72  range: -5.6 to 125.1 mm/mo
  CDL-masked non-NaN : 72  range: -5.6 to 125.1 mm/mo

Northern WA/ID border (lat 48–50N, lon 117–124W):
  Patch shape        : 11×57
  All non-NaN vals   : 49  range: -14.4 to 75.6 mm/mo
  CDL-masked non-NaN : 49  range: -14.4 to 75.6 mm/mo



/tmp/ipykernel_2204395/827257798.py:11: RuntimeWarning: Mean of empty slice
  _all_delta = np.nanmean(np.stack(list(delta_maps.values()), axis=0), axis=0)


### (d) Diagnosis Summary

**Root causes of the three observed problems:**

1. **Border artifacts** — The NLDAS 0.125° grid extends slightly beyond the US political boundary on all sides. The CDL cropland fraction mask was computed from NLDAS-aligned CDL rasters, which inherit these border pixels. Any NLDAS pixel whose center falls within the domain extent (lon −124.69° to −84.06°, lat 25.69° to 49.31°) is retained if ≥50% of the CDL-labeled area is cropland — even if that pixel straddles the Mexican or Canadian border. These border pixels carry real HumanET values, not fill/NoData, so they are rendered as legitimate data. **Fix: apply a hard clip of the masked raster to the dissolved 48-state US boundary after the CDL mask, so any NLDAS pixel whose centre falls outside the US polygon is zeroed to NaN.**

2. **Squished aspect ratio** — All earlier maps used `imshow(..., aspect='auto')` in EPSG:4326 geographic coordinates, which scales 1° longitude = 1° latitude regardless of true distance. At 40°N, 1° longitude ≈ 85 km while 1° latitude ≈ 111 km, so the map is compressed east–west relative to its true shape. **Fix: reproject pixel centres to EPSG:5070 (Albers Equal Area Conic, NAD83) and use `pcolormesh` with equal-aspect axes.**

3. **No state boundaries** — The earlier draft sections used bare `imshow` with no overlay. **Fix: draw state outlines from `us_states_simple.geojson` reprojected to EPSG:5070 as thin `ax.plot` lines.**

All three fixes are applied in Section 4 below.

---

## 4. Publication Figure 1 — CONUS Human ET Maps (Albers Equal Area Conic)

This section produces the two-panel publication figure using **EPSG:5070 Albers Equal Area Conic** projection, fixing three problems visible in the earlier draft maps:

1. **Border artifacts** — the equirectangular display distorts pixels near the CONUS boundary; Albers removes this distortion.
2. **Squished aspect ratio** — 1° lon ≠ 1° lat at mid-latitudes; Albers Equal Area renders CONUS with correct shape and proportions.
3. **No state boundaries** — thin gray state outlines added via a simplified CONUS states GeoJSON.

**Figure 1 panels:**
- **Panel a**: 10-year mean growing-season HumanET (April–September, 2015–2024)
- **Panel b**: Linear trend in growing-season HumanET (mm/month/year, 2015–2024)

**Requirements**: Relies on `delta_maps` (cell 1.3), `crop_mask_static` (cell 1.2), `trend_slope` / `yearly_gs` (cell 2.1) all loaded above.

### 4.1 Albers Projection Setup and State Boundaries

In [13]:
import geopandas as gpd
from pyproj import Transformer
from shapely.geometry import Point, Polygon as _SPoly

# ── EPSG:4326 → EPSG:5070 (Albers Equal Area Conic, NAD83) ───────────────
_t5070 = Transformer.from_crs('EPSG:4326', 'EPSG:5070', always_xy=True)

# ── Projected centre coordinates for pcolormesh (shading='nearest') ───────
_LON_GRID, _LAT_GRID = np.meshgrid(CONUS_LON, CONUS_LAT)   # (189, 325)
_X5070, _Y5070 = _t5070.transform(_LON_GRID, _LAT_GRID)    # (189, 325) metres

# ── Natural Earth boundaries ──────────────────────────────────────────────
# 50m tier, not 110m: at 110m the state polygons are too generalised to
# assign 0.125° pixels reliably (coastal and border cells land outside every
# polygon), which is what produced the border artifacts.
_NE_BASE = ('https://github.com/nvkelso/natural-earth-vector/raw/master/'
            'geojson/')

print('Loading Natural Earth 50m boundaries...')
_ne_states    = gpd.read_file(_NE_BASE + 'ne_50m_admin_1_states_provinces.geojson')
_ne_countries = gpd.read_file(_NE_BASE + 'ne_50m_admin_0_countries.geojson')

_us_states_gdf = _ne_states[
    (_ne_states['admin'] == 'United States of America') &
    (~_ne_states['name'].isin(['Alaska', 'Hawaii']))
][['name', 'geometry']].copy()

# Country-name column differs between Natural Earth tiers.
_cname_col = next(c for c in ['NAME', 'ADMIN', 'name', 'admin']
                  if c in _ne_countries.columns)
_border_countries = _ne_countries[
    _ne_countries[_cname_col].isin(
        ['United States of America', 'Canada', 'Mexico'])
].copy()

print('  {:d} CONUS state polygons loaded'.format(len(_us_states_gdf)))

# ────────────────────────────────────────────────────────────────────────
# Study domain: WHOLE states, snapped
# ────────────────────────────────────────────────────────────────────────
# The eastern limit of the domain is set by OpenET coverage, which stops near
# 84°W.  Cutting the domain at that meridian draws an artificial straight line
# through the middle of several states.  Instead the domain is defined as a set
# of COMPLETE states, so every edge of the study boundary is a real state or
# national border.
#
# A state joins the domain when it (a) contains at least one valid cropland
# pixel and (b) has at least half its area west of the OpenET cutoff.  Rule (b)
# drops Ohio, Georgia and Florida, which each hold only a thin sliver of pixels
# at the extreme eastern edge; keeping them would push the map ~500 km further
# east for 128 pixels (1.4% of the sample).

EAST_CUTOFF_LON     = float(CONUS_LON[-1]) + 0.0625   # -84.0625°W
MIN_STATE_AREA_WEST = 0.50                            # rule (b)

# Fraction of each state's area west of the OpenET cutoff.
_west_halfplane = _SPoly([(-180, 10), (EAST_CUTOFF_LON, 10),
                          (EAST_CUTOFF_LON, 60), (-180, 60)])
_us_states_gdf['frac_west'] = [
    g.intersection(_west_halfplane).area / g.area
    for g in _us_states_gdf.geometry
]

# Which states hold valid cropland pixels?
_cr, _cc = np.where(crop_mask_static)
_crop_pts = gpd.GeoDataFrame(
    {'row': _cr, 'col': _cc},
    geometry=[Point(float(CONUS_LON[c]), float(CONUS_LAT[r]))
              for r, c in zip(_cr, _cc)],
    crs='EPSG:4326',
)
_pt_state = gpd.sjoin(_crop_pts, _us_states_gdf[['name', 'geometry']],
                      how='left', predicate='within')
_pt_state = _pt_state.drop_duplicates(subset=['row', 'col'])
_state_pixel_counts = _pt_state['name'].value_counts()

_us_states_gdf['npix'] = (_us_states_gdf['name']
                          .map(_state_pixel_counts).fillna(0).astype(int))

_study_states_gdf = _us_states_gdf[
    (_us_states_gdf['npix'] >= 1) &
    (_us_states_gdf['frac_west'] >= MIN_STATE_AREA_WEST)
].copy()
STUDY_STATES = sorted(_study_states_gdf['name'].tolist())

print()
print('Study domain: {:d} whole states'.format(len(STUDY_STATES)))
print('  ' + ', '.join(STUDY_STATES))
_excluded = sorted(set(_us_states_gdf[_us_states_gdf['npix'] >= 1]['name'])
                   - set(STUDY_STATES))
print('  excluded (mostly east of OpenET cutoff): '
      + (', '.join(_excluded) if _excluded else 'none'))

# ── Dissolved study polygon, 4326 and 5070 ───────────────────────────────
_study_union_4326 = _study_states_gdf.geometry.union_all()
_study_states_5070 = _study_states_gdf.to_crs('EPSG:5070')
_states_5070       = _us_states_gdf.to_crs('EPSG:5070')
_countries_5070    = _border_countries.to_crs('EPSG:5070')
_study_area_geom   = gpd.GeoSeries([_study_union_4326],
                                   crs='EPSG:4326').to_crs('EPSG:5070').iloc[0]

# ── Pixel mask: cropland pixels inside the study states ──────────────────
# Buffer by half a grid cell so a coastal cell whose CENTRE falls just offshore
# is still retained.
_HALF_CELL_DEG = 0.0625
_study_buffered = _study_union_4326.buffer(_HALF_CELL_DEG)

_boundary_mask = np.zeros((n_lat, n_lon), dtype=bool)
for r, c in zip(_cr, _cc):
    if _study_buffered.contains(Point(float(CONUS_LON[c]), float(CONUS_LAT[r]))):
        _boundary_mask[r, c] = True

n_inside  = int(_boundary_mask.sum())
n_outside = int(crop_mask_static.sum()) - n_inside
print()
print('Pixel mask: {:,} cropland pixels inside study states, '
      '{:,} clipped out'.format(n_inside, n_outside))

# ────────────────────────────────────────────────────────────────────────
# Apply the same clip to the ANALYSIS SAMPLE
# ────────────────────────────────────────────────────────────────────────
# Previously the boundary clip was applied only when drawing maps, so the
# pooled regression and every pixel-level slope still included cells over the
# ocean, Mexico and Canada that had passed the CDL cropland-fraction threshold.
# Those cells are ~98% empty, but the ~2% carrying values entered the fit.
_keep_px = set(
    (round(float(CONUS_LAT[r]), 4), round(float(CONUS_LON[c]), 4))
    for r, c in zip(*np.where(_boundary_mask))
)
_n_rows_before = len(df)
_px_key = list(zip(df['lat'].round(4), df['lon'].round(4)))
df['in_study'] = [k in _keep_px for k in _px_key]

_cc_before = df.dropna(subset=['sif_z', 'delta_et', 'spei90d']).shape[0]
df = df[df['in_study']].drop(columns='in_study').reset_index(drop=True)
_cc_after = df.dropna(subset=['sif_z', 'delta_et', 'spei90d']).shape[0]

print()
print('Analysis sample clipped to study domain:')
print('  panel rows     : {:,} -> {:,}'.format(_n_rows_before, len(df)))
print('  complete cases : {:,} -> {:,}  ({:,} out-of-domain obs removed)'.format(
    _cc_before, _cc_after, _cc_before - _cc_after))

# ── Map extent — tight to the study domain ───────────────────────────────
_sminx, _sminy, _smaxx, _smaxy = _study_area_geom.bounds
_PAD = 12_000     # 12 km — just enough that the boundary stroke is not clipped
# Extra band above the domain so panel labels ("(a)", "SPEI-90d", ...) sit on
# white space instead of colliding with the Canada border and the Washington
# coastline, both of which run along the very top of the frame.
_TOP_PAD = 0.085 * (_smaxy - _sminy)
_xmin, _xmax = _sminx - _PAD, _smaxx + _PAD
_ymin, _ymax = _sminy - _PAD, _smaxy + _TOP_PAD
print()
print('Map extent (EPSG:5070): x {:,.0f} to {:,.0f} | y {:,.0f} to {:,.0f}'.format(
    _xmin, _xmax, _ymin, _ymax))


# ────────────────────────────────────────────────────────────────────────
# Basemap helpers
# ────────────────────────────────────────────────────────────────────────

def _add_basemap(ax, us_color='white'):
    """White page, white land; state strokes are drawn separately."""
    ax.set_facecolor('white')
    _states_5070.plot(ax=ax, color=us_color, edgecolor='none', zorder=1)
    ax.set_xlim(_xmin, _xmax)
    ax.set_ylim(_ymin, _ymax)


def _draw_states(ax, lw=0.4, color='#333333', zorder=5):
    """Overlay state and country boundaries."""
    _states_5070.boundary.plot(ax=ax, color=color, linewidth=lw, zorder=zorder)
    _countries_5070.boundary.plot(ax=ax, color='#111111', linewidth=lw * 2,
                                  zorder=zorder + 1)


def _draw_study_boundary(ax, lw=2.0, color='#111111', zorder=12):
    """Bold outline of the study domain.

    Every segment is a real state or national border — the domain is a union
    of complete state polygons, so there is no artificial straight edge.
    """
    _bnd = _study_area_geom.boundary
    _geoms = list(_bnd.geoms) if hasattr(_bnd, 'geoms') else [_bnd]
    for _g in _geoms:
        ax.plot(*_g.xy, color=color, linewidth=lw, zorder=zorder,
                solid_capstyle='round', solid_joinstyle='round')


def _albers_axes(ax):
    """Remove ticks, spines, labels; set equal-aspect Albers limits."""
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_aspect('equal')
    ax.set_xlim(_xmin, _xmax)
    ax.set_ylim(_ymin, _ymax)


def _map_fig(figsize=(9.5, 6.0)):
    """Standard single-map figure."""
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    fig.patch.set_facecolor('white')
    return fig, ax


Loading Natural Earth 50m boundaries...


  49 CONUS state polygons loaded

Study domain: 30 whole states
  Alabama, Arizona, Arkansas, California, Colorado, Idaho, Illinois, Indiana, Iowa, Kansas, Kentucky, Louisiana, Michigan, Minnesota, Mississippi, Missouri, Montana, Nebraska, Nevada, New Mexico, North Dakota, Oklahoma, Oregon, South Dakota, Tennessee, Texas, Utah, Washington, Wisconsin, Wyoming
  excluded (mostly east of OpenET cutoff): Florida, Georgia, Ohio



Pixel mask: 9,080 cropland pixels inside study states, 16,073 clipped out



Analysis sample clipped to study domain:
  panel rows     : 1,509,180 -> 544,800
  complete cases : 434,245 -> 419,721  (14,524 out-of-domain obs removed)

Map extent (EPSG:5070): x -2,365,236 to 1,278,086 | y 302,503 to 3,414,387


### 4.2 Compute 10-Year Mean Growing-Season HumanET

Average all 60 growing-season monthly HumanET maps (April–September, 2015–2024). Pixels with fewer than 30 valid months (50% of 60) are masked to NaN to avoid biasing the mean toward years with sparser sampling.

In [14]:
# ── Stack all 60 growing-season HumanET maps ─────────────────────────────
_all_keys = [(y, m) for y in YEARS for m in GROWING_SEASON if (y, m) in delta_maps]
_stack    = np.stack([delta_maps[k] for k in _all_keys], axis=0)   # (60, 189, 325)

# ── Mean and valid-count ───────────────────────────────────────────────────
_n_valid      = np.sum(np.isfinite(_stack), axis=0)                 # (189, 325)
_human_et_mean = np.nanmean(_stack, axis=0)                         # (189, 325)

# Require at least 30 valid months (50%) for a reliable mean
MIN_MONTHS = 30
_human_et_mean[_n_valid < MIN_MONTHS] = np.nan

# Restrict to cropland pixels only
_human_et_mean[~crop_mask_static] = np.nan

# ── Summary statistics ─────────────────────────────────────────────────────
_valid_mean = _human_et_mean[np.isfinite(_human_et_mean)]
print('Mean HumanET summary (cropland pixels only):')
print('  N valid pixels : {:,}'.format(len(_valid_mean)))
print('  Mean           : {:.1f} mm/month'.format(float(_valid_mean.mean())))
print('  5th percentile : {:.1f} mm/month'.format(float(np.percentile(_valid_mean, 5))))
print('  95th percentile: {:.1f} mm/month'.format(float(np.percentile(_valid_mean, 95))))
print('  Max            : {:.1f} mm/month'.format(float(_valid_mean.max())))

# Color scale: 0 to 95th percentile
_mean_vmax = float(np.percentile(_valid_mean[_valid_mean > 0], 95)) if (_valid_mean > 0).any() else 50.0
print('  Color scale vmax (95th pct): {:.1f} mm/month'.format(_mean_vmax))

Mean HumanET summary (cropland pixels only):
  N valid pixels : 7,578
  Mean           : 22.4 mm/month
  5th percentile : 1.5 mm/month
  95th percentile: 59.3 mm/month
  Max            : 142.8 mm/month
  Color scale vmax (95th pct): 60.2 mm/month


/tmp/ipykernel_2204395/4102909592.py:7: RuntimeWarning: Mean of empty slice
  _human_et_mean = np.nanmean(_stack, axis=0)                         # (189, 325)


### 4.3 Publication Figure 1: Two-Panel Albers Maps

Combined publication figure saved to `figures/conus/` as both **PNG** (300 DPI) and **PDF** (vector).

Colormap choices:
- Panel a (mean HumanET): `YlOrBr` — warm sequential, suitable for positive-only irrigation intensity
- Panel b (trend): `RdBu` — diverging, blue = increasing HumanET, red = decreasing

In [ ]:
import matplotlib.ticker as mticker

# ── Figure 1a: 10-year mean growing-season Human ET ──────────────────────
# Two colour variants of the same map:
#   default   — ltc 'ploen' reversed (high = blue)
#   heatmap0  — ltc 'heatmap0'

_final_mask = _boundary_mask if _boundary_mask is not None else crop_mask_static
_mean_plot  = np.where(_final_mask, _human_et_mean, np.nan)

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 22,
    'axes.labelsize': 19, 'axes.titlesize': 19,
    'xtick.labelsize': 18, 'ytick.labelsize': 18,
    'legend.fontsize': 12,
})

_FIG01A_VARIANTS = [
    (cmap_ploen_r_map,  'fig01a_human_et_mean'),
    (cmap_heatmap0_map, 'fig01a_human_et_mean_heatmap0'),
]

for _cmap_a, _fname_a in _FIG01A_VARIANTS:
    fig, ax = _map_fig()

    im = ax.pcolormesh(
        _X5070, _Y5070, _mean_plot,
        cmap=_cmap_a, vmin=0, vmax=_mean_vmax,
        shading='nearest', rasterized=True, zorder=2,
    )
    _albers_axes(ax)
    _add_basemap(ax)
    _draw_states(ax, lw=0.4, color='#333333', zorder=5)
    _draw_study_boundary(ax, lw=2.0, zorder=10)

    cb = fig.colorbar(im, ax=ax, orientation='horizontal',
                      fraction=0.04, pad=0.03, shrink=0.72)
    cb.set_label('Mean Human ET [mm month⁻¹]', fontsize=17)
    cb.ax.tick_params(labelsize=15)

    ax.text(0.015, 0.97, '(a)', transform=ax.transAxes,
            fontsize=28, fontweight='bold', va='top', ha='left', zorder=15,
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                      edgecolor='none', alpha=1.0))

    plt.tight_layout()
    plt.savefig(str(figs / _fname_a) + '.png', dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.savefig(str(figs / _fname_a) + '.pdf', bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.show()
    print('Saved:', _fname_a + '.png / .pdf')


In [ ]:
# ── Figure 1b: Trend in growing-season Human ET ──────────────────────────
# Two colour variants:
#   default   — RdBu (positive = blue, negative = red)
#   heatmap0  — ltc 'heatmap0'

_finite_slopes = trend_slope[_final_mask & np.isfinite(trend_slope)]
_trend_vmax    = max(float(np.nanpercentile(np.abs(_finite_slopes), 97.5)), 0.10)
_trend_plot    = np.where(_final_mask, trend_slope, np.nan)

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 22,
    'axes.labelsize': 19, 'axes.titlesize': 19,
    'xtick.labelsize': 18, 'ytick.labelsize': 18,
    'legend.fontsize': 12,
})

_FIG01B_VARIANTS = [
    (cmap_rdbu_map,     'fig01b_human_et_trend'),
    (cmap_heatmap0_map, 'fig01b_human_et_trend_heatmap0'),
]

for _cmap_b, _fname_b in _FIG01B_VARIANTS:
    fig, ax = _map_fig()

    im = ax.pcolormesh(
        _X5070, _Y5070, _trend_plot,
        cmap=_cmap_b, vmin=-_trend_vmax, vmax=_trend_vmax,
        shading='nearest', rasterized=True, zorder=2,
    )
    _albers_axes(ax)
    _add_basemap(ax)
    _draw_states(ax, lw=0.4, color='#333333', zorder=5)
    _draw_study_boundary(ax, lw=2.0, zorder=10)

    cb = fig.colorbar(im, ax=ax, orientation='horizontal',
                      fraction=0.04, pad=0.03, shrink=0.72)
    cb.set_label('Trend [mm month⁻¹ yr⁻¹]', fontsize=17)
    cb.ax.tick_params(labelsize=15)
    _tk = [-_trend_vmax, -_trend_vmax / 2, 0, _trend_vmax / 2, _trend_vmax]
    cb.set_ticks(_tk)
    cb.set_ticklabels(['{:.1f}'.format(v) for v in _tk])

    ax.text(0.015, 0.97, '(b)', transform=ax.transAxes,
            fontsize=28, fontweight='bold', va='top', ha='left', zorder=15,
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                      edgecolor='none', alpha=1.0))

    plt.tight_layout()
    plt.savefig(str(figs / _fname_b) + '.png', dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.savefig(str(figs / _fname_b) + '.pdf', bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.show()
    print('Saved:', _fname_b + '.png / .pdf')


In [17]:
from matplotlib.lines import Line2D

# ── Shared helpers for the Figure 1 scatter panels (1c and 1d) ───────────
# Density is conveyed by overplotting: every point is drawn with a low alpha,
# so regions where many pixels coincide accumulate to a darker colour.  This
# replaces the previous solid "blob" of opaque markers.

CA_COLOR = PALETTE_PLOEN[4]   # '#B17776' warm brownish red
IA_COLOR = PALETTE_PLOEN[0]   # '#3F5671' dark blue
OTHER_COLOR = '#9E9E9E'

# alpha, size, z-order per group — tuned so 'other' reads as a density cloud
_SCATTER_STYLE = {
    'other': dict(color=OTHER_COLOR, s=7,  alpha=0.10, zorder=3, label='Other states'),
    'CA':    dict(color=CA_COLOR,    s=13, alpha=0.30, zorder=5, label='California'),
    'IA':    dict(color=IA_COLOR,    s=13, alpha=0.30, zorder=5, label='Iowa'),
}


def _state_label_for_lonlat(lons, lats):
    """Label each (lon, lat) point as 'CA', 'IA' or 'other' via spatial join."""
    lons = np.asarray(lons, dtype=float)
    lats = np.asarray(lats, dtype=float)
    _pts = gpd.GeoDataFrame(
        {'_i': np.arange(len(lons))},
        geometry=[Point(x, y) for x, y in zip(lons, lats)],
        crs='EPSG:4326',
    )
    _j = gpd.sjoin(_pts, _us_states_gdf[['name', 'geometry']],
                   how='left', predicate='within')
    _j = _j.drop_duplicates(subset=['_i']).sort_values('_i')
    _name = _j['name'].values
    return np.where(_name == 'California', 'CA',
                    np.where(_name == 'Iowa', 'IA', 'other'))


def _state_label_for_pixels(rows, cols):
    """Label each (row, col) grid pixel as 'CA', 'IA' or 'other'."""
    return _state_label_for_lonlat(
        [float(CONUS_LON[c]) for c in cols],
        [float(CONUS_LAT[r]) for r in rows],
    )


def _draw_fit(ax, x, y, color, min_n=25):
    """Least-squares fit line drawn across the observed x-range."""
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < min_n:
        return None
    slope, intercept, r, p, se = stats.linregress(x[ok], y[ok])
    xs = np.linspace(np.nanmin(x[ok]), np.nanmax(x[ok]), 100)
    ax.plot(xs, intercept + slope * xs, color=color, linewidth=3.0,
            zorder=8, solid_capstyle='round')
    # White casing underneath so the line stays legible over dense points
    ax.plot(xs, intercept + slope * xs, color='white', linewidth=5.0,
            zorder=7, solid_capstyle='round')
    return slope, intercept, r, p, se


def _scatter_panel(ax, df_pts, xcol, ycol, xlabel, ylabel, panel_letter):
    """Shared scatter rendering for fig01c / fig01d."""
    for key in ['other', 'CA', 'IA']:
        sub = df_pts[df_pts['state'] == key]
        st  = _SCATTER_STYLE[key]
        ax.scatter(sub[xcol], sub[ycol], c=st['color'], s=st['s'],
                   alpha=st['alpha'], linewidths=0, zorder=st['zorder'],
                   rasterized=True)

    ax.axhline(0, color='#444444', lw=1.2, linestyle='--', zorder=2)

    fits = {}
    for key, color in [('CA', CA_COLOR), ('IA', IA_COLOR)]:
        sub = df_pts[df_pts['state'] == key]
        fit = _draw_fit(ax, sub[xcol].values, sub[ycol].values, color)
        if fit is not None:
            fits[key] = fit

    ax.set_xlabel(xlabel, fontsize=19)
    ax.set_ylabel(ylabel, fontsize=19)
    ax.tick_params(labelsize=16)
    ax.grid(alpha=0.2)

    # Legend proxies at full opacity (the plotted points are deliberately faint)
    _handles = [
        Line2D([], [], marker='o', linestyle='none', markersize=9,
               markerfacecolor=_SCATTER_STYLE[k]['color'],
               markeredgecolor='none', label=_SCATTER_STYLE[k]['label'])
        for k in ['other', 'CA', 'IA']
    ]
    ax.legend(handles=_handles, fontsize=15, framealpha=0.92,
              loc='upper right', borderpad=0.5, handletextpad=0.4)

    ax.text(0.015, 0.97, panel_letter, transform=ax.transAxes,
            fontsize=28, fontweight='bold', va='top', ha='left')
    return fits


def _report_fits(fits, label):
    for key, (slope, intercept, r, p, se) in fits.items():
        print('  {:12s} {:3s}: slope = {:+.4f}  r = {:+.3f}  p = {:.3g}'.format(
            label, key, slope, r, p))


In [18]:
# ── Figure 1c: Scatter — 10-yr mean Human ET vs trend ────────────────────
_rows_c, _cols_c = np.where(_final_mask & np.isfinite(_human_et_mean)
                            & np.isfinite(trend_slope))

df_sc = pd.DataFrame({
    'mean_et': _human_et_mean[_rows_c, _cols_c],
    'trend':   trend_slope[_rows_c, _cols_c],
    'state':   _state_label_for_pixels(_rows_c, _cols_c),
})

print('Figure 1c pixels: {:,} total | CA {:,} | Iowa {:,} | other {:,}'.format(
    len(df_sc),
    int((df_sc['state'] == 'CA').sum()),
    int((df_sc['state'] == 'IA').sum()),
    int((df_sc['state'] == 'other').sum()),
))

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 22,
    'axes.labelsize': 19, 'axes.titlesize': 19,
    'xtick.labelsize': 16, 'ytick.labelsize': 16,
    'legend.fontsize': 15,
})

fig, ax = plt.subplots(figsize=(8.5, 6.5))
fig.patch.set_facecolor('white')

_fits_c = _scatter_panel(
    ax, df_sc, 'mean_et', 'trend',
    '10-yr mean growing-season Human ET [mm month⁻¹]',
    'Trend in Human ET [mm month⁻¹ yr⁻¹]',
    '(c)',
)
_report_fits(_fits_c, 'fig01c')

plt.tight_layout()
_fig01c = figs / 'fig01c_mean_vs_trend_scatter'
plt.savefig(str(_fig01c) + '.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.savefig(str(_fig01c) + '.pdf', bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print('Saved: fig01c_mean_vs_trend_scatter.png / .pdf')


Figure 1c pixels: 7,330 total | CA 223 | Iowa 804 | other 6,303
  fig01c       CA : slope = -0.0026  r = -0.038  p = 0.574
  fig01c       IA : slope = +0.0095  r = +0.086  p = 0.0145


Saved: fig01c_mean_vs_trend_scatter.png / .pdf


In [19]:
# ── Figure 1d: First-year Human ET vs trend ──────────────────────────────
# Panel (d) of Figure 1.  x-axis is the FIRST year of record (2015) growing-
# season mean Human ET, so the panel reads as "where irrigation started high,
# which way has it moved since?"  Same CA / Iowa case-study colouring and
# density treatment as panel (c).

_first_year   = YEARS[0]
_first_year_et = gs_stack[0]          # (n_lat, n_lon) — YEARS[0] GS mean

_rows_d, _cols_d = np.where(_final_mask & np.isfinite(_first_year_et)
                            & np.isfinite(trend_slope))

df_sd = pd.DataFrame({
    'first_et': _first_year_et[_rows_d, _cols_d],
    'trend':    trend_slope[_rows_d, _cols_d],
    'state':    _state_label_for_pixels(_rows_d, _cols_d),
})

print('Figure 1d pixels ({} baseline): {:,} total | CA {:,} | Iowa {:,}'.format(
    _first_year, len(df_sd),
    int((df_sd['state'] == 'CA').sum()),
    int((df_sd['state'] == 'IA').sum()),
))

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 22,
    'axes.labelsize': 19, 'axes.titlesize': 19,
    'xtick.labelsize': 16, 'ytick.labelsize': 16,
    'legend.fontsize': 15,
})

fig, ax = plt.subplots(figsize=(8.5, 6.5))
fig.patch.set_facecolor('white')

_fits_d = _scatter_panel(
    ax, df_sd, 'first_et', 'trend',
    '{} mean growing-season Human ET [mm month⁻¹]'.format(_first_year),
    'Trend in Human ET [mm month⁻¹ yr⁻¹]',
    '(d)',
)
_report_fits(_fits_d, 'fig01d')

# Robust symmetric y-limits: a handful of extreme trends stretched the axis to
# -20..+12 and squashed the actual cloud into a thin band.
_ylim_d = float(np.nanpercentile(np.abs(df_sd['trend'].values), 99.5))
ax.set_ylim(-_ylim_d, _ylim_d)

plt.tight_layout()
_fig01d = figs / 'fig01d_firstyear_vs_trend_scatter'
plt.savefig(str(_fig01d) + '.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.savefig(str(_fig01d) + '.pdf', bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print('Saved: fig01d_firstyear_vs_trend_scatter.png / .pdf')


Figure 1d pixels (2015 baseline): 7,345 total | CA 223 | Iowa 804
  fig01d       CA : slope = -0.0196  r = -0.316  p = 1.41e-06
  fig01d       IA : slope = -0.0405  r = -0.452  p = 8.43e-42


Saved: fig01d_firstyear_vs_trend_scatter.png / .pdf


In [20]:
# ── Supplemental Table: per-state min / median / max of mean ET and trend ─
# Only states with > 25 valid cropland pixels included.
# Assignment is a single vectorised spatial join (the previous version looped
# over every pixel x every state polygon, which took minutes).

_rows_s, _cols_s = np.where(_final_mask & np.isfinite(_human_et_mean)
                            & np.isfinite(trend_slope))

_supp_pts = gpd.GeoDataFrame(
    {'_i': np.arange(len(_rows_s)),
     'mean_et': _human_et_mean[_rows_s, _cols_s],
     'trend':   trend_slope[_rows_s, _cols_s]},
    geometry=[Point(float(CONUS_LON[c]), float(CONUS_LAT[r]))
              for r, c in zip(_rows_s, _cols_s)],
    crs='EPSG:4326',
)
_supp_j = gpd.sjoin(_supp_pts, _us_states_gdf[['name', 'geometry']],
                    how='left', predicate='within')
_supp_j = _supp_j.drop_duplicates(subset=['_i'])

df_state_pix = pd.DataFrame({
    'state':   _supp_j['name'].fillna('Unknown').values,
    'mean_et': _supp_j['mean_et'].values,
    'trend':   _supp_j['trend'].values,
})

df_supp = (
    df_state_pix.groupby('state')
    .agg(n_pixels=('mean_et', 'size'),
         mean_et_min=('mean_et', 'min'),
         mean_et_median=('mean_et', 'median'),
         mean_et_max=('mean_et', 'max'),
         trend_min=('trend', 'min'),
         trend_median=('trend', 'median'),
         trend_max=('trend', 'max'))
    .reset_index()
)
df_supp = df_supp[(df_supp['n_pixels'] > 25) & (df_supp['state'] != 'Unknown')]
df_supp = df_supp.sort_values('state').reset_index(drop=True)

print('States with > 25 valid cropland pixels: {:d}'.format(len(df_supp)))
print(df_supp.to_string(index=False,
      float_format=lambda x: '{:.2f}'.format(x) if abs(x) < 1000 else '{:.0f}'.format(x)))

_supp_path = figs / 'supp_table_state_stats.csv'
df_supp.to_csv(_supp_path, index=False, float_format='%.3f')
print()
print('Saved: supp_table_state_stats.csv')


States with > 25 valid cropland pixels: 22
       state  n_pixels  mean_et_min  mean_et_median  mean_et_max  trend_min  trend_median  trend_max
    Arkansas       191       -14.25            5.09        50.33      -2.21          0.70       2.10
  California       223        -2.00           77.48       123.97      -5.25          0.58       6.49
    Colorado       186        -3.62           14.24        76.05      -1.63          0.81       3.01
       Idaho       121        -5.20           48.81        81.48      -1.93          0.58       3.11
    Illinois       648       -12.28            6.37        39.49      -1.62          0.65       3.85
        Iowa       804       -10.80           10.27        39.77      -0.54          1.16       3.37
      Kansas       610         3.41           17.77        65.72      -0.66          1.72       5.20
    Kentucky        56       -18.29            8.32        23.10      -0.65          0.57       2.20
   Louisiana        99       -21.99            3

---

## 5. Figure A — Per-Pixel Irrigation Buffering: Map + Scatter (Figure 2 Candidate)

This figure addresses the question: *Where* does irrigation buffer SIF under drought, and do heavily irrigated pixels show stronger buffering?

**Panel a (map):** Per-pixel OLS slope of SIF z-score ~ HumanET estimated using drought months only (SPEI ≤ −0.5, p < 0.10). Blue = positive buffering effect; red = no benefit. EPSG:5070 Albers projection with state outlines.

**Panel b (scatter):** x-axis = 10-year mean HumanET at each significant pixel (proxy for irrigation intensity); y-axis = pixel-level slope. Color = latitude (proxy for climate regime). LOWESS smoothing line shows the overall trend: do more heavily irrigated regions have stronger irrigation buffering?

### 5.1 Build Per-Pixel Data for Scatter

In [21]:
# ── Per-drought-category pixel regressions (for Figure 2 four-map layout) ─
# For each drought severity category, run OLS sif_z ~ delta_et per pixel.
# Results stored as: pix_slopes_by_cat[cat_label] = DataFrame(lat, lon, slope, pval, n)

_CAT_DEFS_FIG2 = [
    ('No Drought\n(SPEI>-0.5)',         df['spei90d'] >  -0.5),
    ('Mild\n(-1.0<SPEI≤-0.5)',   (df['spei90d'] > -1.0) & (df['spei90d'] <= -0.5)),
    ('Moderate\n(-1.5<SPEI≤-1.0)', (df['spei90d'] > -1.5) & (df['spei90d'] <= -1.0)),
    ('Severe\n(SPEI≤-1.5)',          df['spei90d'] <= -1.5),
]

_CAT_FILENAMES_FIG2 = [
    'fig02_slope_no_drought',
    'fig02_slope_mild',
    'fig02_slope_moderate',
    'fig02_slope_severe',
]

def _pixel_slope_series(grp):
    xy = grp[['delta_et', 'sif_z']].dropna()
    if len(xy) < MIN_OBS_PIXEL:
        return pd.Series({'slope': np.nan, 'pval': np.nan, 'n': len(xy)})
    slope, _, _, pval, _ = stats.linregress(xy['delta_et'].values, xy['sif_z'].values)
    return pd.Series({'slope': slope, 'pval': pval, 'n': len(xy)})

_df_reg = df.dropna(subset=['sif_z', 'spei90d', 'delta_et']).copy()
_df_reg = _df_reg[np.isfinite(_df_reg['sif_z']) & np.isfinite(_df_reg['spei90d'])
                  & np.isfinite(_df_reg['delta_et'])].copy()

pix_slopes_by_cat = {}

for cat_name, cat_mask in _CAT_DEFS_FIG2:
    _df_cat = _df_reg[cat_mask.reindex(_df_reg.index, fill_value=False)].copy()
    print('  {:35s} | {:,} obs'.format(cat_name.replace('\n', ' '), len(_df_cat)))
    _stats = (_df_cat.groupby(['lat', 'lon']).apply(_pixel_slope_series).reset_index())
    _sig   = _stats[_stats['pval'] < ALPHA_SPATIAL].copy()
    pix_slopes_by_cat[cat_name] = _sig
    n_pos = int((_sig['slope'] > 0).sum())
    n_neg = int((_sig['slope'] < 0).sum())
    print('    sig pixels: {:,}  | pos: {:,}  neg: {:,}'.format(len(_sig), n_pos, n_neg))

print('\nPer-category regressions complete.')

  No Drought (SPEI>-0.5)              | 304,925 obs


    sig pixels: 2,688  | pos: 2,000  neg: 688
  Mild (-1.0<SPEI≤-0.5)               | 64,544 obs


    sig pixels: 1,395  | pos: 1,175  neg: 220
  Moderate (-1.5<SPEI≤-1.0)           | 36,776 obs


    sig pixels: 688  | pos: 595  neg: 93
  Severe (SPEI≤-1.5)                  | 13,476 obs


    sig pixels: 68  | pos: 50  neg: 18

Per-category regressions complete.


In [22]:
# ── Per-pixel mean HumanET and corn fraction ──────────────────────────────
# Fix: pandas groupby with multiple keys returns (key_tuple, group), not 3-tuple.
_pix_mean_et   = {}
_pix_corn_frac = {}

for (px_lat, px_lon), grp in df.groupby(['lat', 'lon']):
    _vals = grp['delta_et'].dropna()
    if len(_vals) >= 3:
        _pix_mean_et[(px_lat, px_lon)] = float(_vals.mean())
    if 'corn_frac' in grp.columns:
        _cf = grp['corn_frac'].dropna()
        if len(_cf) >= 1:
            _pix_corn_frac[(px_lat, px_lon)] = float(_cf.mean())

# ── Build scatter DataFrame from pix_sig (significant drought-month slopes) ─
_scatter_rows = []
for _, row in pix_sig.iterrows():
    _key = (round(float(row['lat']), 4), round(float(row['lon']), 4))
    _mean_et   = _pix_mean_et.get(_key, np.nan)
    _corn_frac = _pix_corn_frac.get(_key, np.nan)
    _scatter_rows.append({
        'lat':       float(row['lat']),
        'lon':       float(row['lon']),
        'slope':     float(row['slope']),
        'pval':      float(row['pval']),
        'n_obs':     int(row['n']),
        'mean_et':   _mean_et,
        'corn_frac': _corn_frac,
    })

df_scatter = pd.DataFrame(_scatter_rows)
df_scatter = df_scatter.dropna(subset=['mean_et', 'slope'])
df_scatter = df_scatter[(df_scatter['mean_et'] > -20) & (df_scatter['mean_et'] < 250)]

print('Scatter dataset: {:,} significant pixels with valid mean HumanET'.format(len(df_scatter)))
print('  Positive slope (buffering): {:,} ({:.0f}%)'.format(
    (df_scatter['slope'] > 0).sum(),
    100 * (df_scatter['slope'] > 0).mean(),
))
print('  Negative slope:             {:,} ({:.0f}%)'.format(
    (df_scatter['slope'] < 0).sum(),
    100 * (df_scatter['slope'] < 0).mean(),
))
print('  Mean HumanET range: {:.1f} to {:.1f} mm/month'.format(
    df_scatter['mean_et'].min(), df_scatter['mean_et'].max()))
print('  corn_frac available: {} pixels'.format(df_scatter['corn_frac'].notna().sum()))

Scatter dataset: 1,977 significant pixels with valid mean HumanET
  Positive slope (buffering): 1,649 (83%)
  Negative slope:             328 (17%)
  Mean HumanET range: -13.4 to 127.4 mm/month
  corn_frac available: 0 pixels


### 5.2 Figure A: Slope Map + Scatter

In [ ]:
# ── Figure 2: Four separate slope maps — one per drought category ──────────
# ltc 'ploen' reversed (positive slope = blue). Significant pixels only.
# Files: fig02_slope_{no_drought,mild,moderate,severe}

from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 22,
    'axes.labelsize': 19, 'axes.titlesize': 19,
    'xtick.labelsize': 18, 'ytick.labelsize': 18,
    'legend.fontsize': 12,
})

_use_final_mask = _boundary_mask if _boundary_mask is not None else crop_mask_static
_lat_to_row = {round(float(v), 4): i for i, v in enumerate(CONUS_LAT)}
_lon_to_col = {round(float(v), 4): i for i, v in enumerate(CONUS_LON)}

# Shared symmetric color range across all 4 maps (±97.5th pct of all sig slopes)
_all_slopes = np.concatenate([
    pix_slopes_by_cat[cn]['slope'].dropna().values
    for cn in pix_slopes_by_cat
])
_sv_max = max(float(np.percentile(np.abs(_all_slopes), 97.5)), 0.005)

# Display title + output filename, in the same order as _CAT_DEFS_FIG2.
# The dict KEY is taken from _CAT_DEFS_FIG2 rather than re-typed here: an
# earlier version hand-copied the keys and dropped a minus sign in the Mild and
# Moderate labels, so pix_slopes_by_cat.get() silently returned empty frames
# and both panels rendered blank despite having 1,395 and 688 significant
# pixels. Deriving the keys removes that failure mode entirely.
_CAT_DISPLAY = [
    ('SPEI > −0.5',              'fig02_slope_no_drought'),
    ('−1.0 < SPEI ≤ −0.5', 'fig02_slope_mild'),
    ('−1.5 < SPEI ≤ −1.0', 'fig02_slope_moderate'),
    ('SPEI ≤ −1.5',            'fig02_slope_severe'),
]
_CAT_LABELS = [
    (_defn[0], _title, _fname)
    for _defn, (_title, _fname) in zip(_CAT_DEFS_FIG2, _CAT_DISPLAY)
]

_missing = [k for k, _, _ in _CAT_LABELS if k not in pix_slopes_by_cat]
assert not _missing, 'category keys absent from pix_slopes_by_cat: {}'.format(_missing)
print('Figure 2 categories resolved:')
for _k, _t, _f in _CAT_LABELS:
    print('  {:22s} -> {:,} sig. pixels'.format(_t, len(pix_slopes_by_cat[_k])))

for (cat_key, cat_title, fname) in _CAT_LABELS:
    pix_cat = pix_slopes_by_cat.get(cat_key, pd.DataFrame())

    _slope_grid = np.full((n_lat, n_lon), np.nan)
    for _, row in pix_cat.iterrows():
        ri = _lat_to_row.get(round(float(row['lat']), 4))
        ci = _lon_to_col.get(round(float(row['lon']), 4))
        if ri is not None and ci is not None:
            _slope_grid[ri, ci] = row['slope']

    # Three states to distinguish, so two layers are needed:
    #   significant slope        -> colour ramp
    #   cropland, not significant -> grey
    #   everything else           -> white (basemap shows through)
    # A single layer with set_bad('#888888') cannot do this: every
    # non-cropland cell in the array is NaN too, so the grey floods the whole
    # 189x325 rectangle.
    _sig_plot = np.where(_use_final_mask & np.isfinite(_slope_grid),
                         _slope_grid, np.nan)
    _nonsig_mask = _use_final_mask & ~np.isfinite(_slope_grid)

    _cmap_s = cmap_ploen_r_map.copy()
    _cmap_s.set_bad('white', alpha=0.0)

    fig, ax = _map_fig()

    # Grey layer first, then the significant slopes on top.
    ax.pcolormesh(
        _X5070, _Y5070,
        np.ma.masked_where(~_nonsig_mask, np.zeros_like(_slope_grid)),
        cmap=ListedColormap(['#888888']), vmin=0, vmax=1,
        shading='nearest', rasterized=True, zorder=2,
    )
    im = ax.pcolormesh(
        _X5070, _Y5070, _sig_plot,
        cmap=_cmap_s, vmin=-_sv_max, vmax=_sv_max,
        shading='nearest', rasterized=True, zorder=3,
    )
    _albers_axes(ax)
    _add_basemap(ax)
    _draw_states(ax, lw=0.4, color='#333333', zorder=5)
    _draw_study_boundary(ax, lw=2.0, zorder=10)

    cb = fig.colorbar(im, ax=ax, orientation='horizontal',
                      fraction=0.04, pad=0.03, shrink=0.72)
    # Shorter label: the full wording overflowed the figure width.
    cb.set_label('SIF z-score per mm month⁻¹ Human ET', fontsize=15)
    cb.ax.tick_params(labelsize=14)
    cb.set_ticks([-_sv_max, 0, _sv_max])
    cb.set_ticklabels(['{:.3f}'.format(-_sv_max), '0', '{:.3f}'.format(_sv_max)])

    _leg = [Patch(facecolor='#888888', edgecolor='none',
                  label='Not significant / < {:d} obs'.format(MIN_OBS_PIXEL))]
    _lg = ax.legend(handles=_leg, loc='lower left', fontsize=14,
                    framealpha=1.0, handlelength=1.2, borderpad=0.5)
    _lg.set_zorder(15)

    ax.text(0.015, 0.97, cat_title, transform=ax.transAxes,
            fontsize=20, fontweight='bold', va='top', ha='left', zorder=15,
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=1.0,
                      edgecolor='none'))

    ax.text(0.99, 0.03, 'n={:,} sig. pixels'.format(len(pix_cat)),
            transform=ax.transAxes, fontsize=13, ha='right', va='bottom',
            color='#555555')

    plt.tight_layout()
    plt.savefig(str(figs / fname) + '.png', dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.savefig(str(figs / fname) + '.pdf', bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.show()
    print('Saved:', fname + '.png / .pdf')

print()
print('Figure 2 complete: 4 maps exported (SPEI-90d).')


In [ ]:
# ── Figure 2b: Bivariate maps — buffering slope x drought severity ────────
# Each pixel is classified into terciles on two independent axes and given one
# of 9 colours from a 2-D grid:
#
#   x-axis : per-pixel OLS slope of SIF z-score on Human ET  (buffering)
#   y-axis : per-pixel mean drought index, inverted so higher = drier
#
# A 1-D ramp such as heatmap0 cannot encode two axes, so the grid is built by
# bilinear interpolation in RGB between four corner colours drawn from
# heatmap0.  Each axis then varies independently and the legend reads as a
# proper 3x3 square.

from matplotlib.colors import to_rgb, to_hex, ListedColormap, BoundaryNorm

# Corner colours (all from PALETTE_HEATMAP0, plus a light neutral tint)
_BIV_C00 = '#EDE7D9'   # low slope,  low drought  — light neutral (E9D8A6 tint)
_BIV_C10 = '#0A9396'   # high slope, low drought  — teal
_BIV_C01 = '#AE2012'   # low slope,  high drought — red
_BIV_C11 = '#001219'   # high slope, high drought — near-black


def _bivariate_grid(c00, c10, c01, c11, n=3):
    """Bilinear RGB blend of four corners into an n x n colour grid."""
    a, b, c, d = (np.array(to_rgb(x)) for x in (c00, c10, c01, c11))
    grid = np.empty((n, n), dtype=object)
    for j in range(n):                       # j = drought tercile (y)
        fy = j / (n - 1)
        for i in range(n):                   # i = slope tercile (x)
            fx = i / (n - 1)
            rgb = (a * (1 - fx) * (1 - fy) + b * fx * (1 - fy)
                   + c * (1 - fx) * fy + d * fx * fy)
            grid[j, i] = to_hex(np.clip(rgb, 0, 1))
    return grid


BIV_GRID = _bivariate_grid(_BIV_C00, _BIV_C10, _BIV_C01, _BIV_C11, n=3)
print('Bivariate 3x3 colour grid (rows = drought tercile, cols = slope tercile):')
for j in range(3):
    print('  ' + '  '.join(BIV_GRID[j, i] for i in range(3)))


def _add_bivariate_legend(fig, grid, xlabel, ylabel,
                          rect=(0.085, 0.17, 0.155, 0.155)):
    """Draw the 3x3 key as an inset axes with arrow labels on both axes."""
    lax = fig.add_axes(rect)
    n = grid.shape[0]
    for j in range(n):
        for i in range(n):
            lax.add_patch(plt.Rectangle((i, j), 1, 1, facecolor=grid[j, i],
                                        edgecolor='white', linewidth=1.2))
    lax.set_xlim(0, n)
    lax.set_ylim(0, n)
    lax.set_xticks([])
    lax.set_yticks([])
    for s in lax.spines.values():
        s.set_visible(False)
    lax.set_aspect('equal')
    # The labels already carry an arrow glyph, so the separate annotate
    # arrows are redundant — and they were what overlapped in the corner.
    lax.set_xlabel(xlabel, fontsize=13, labelpad=6)
    lax.set_ylabel(ylabel, fontsize=13, labelpad=6)
    return lax


def _pixel_slope_table(df_in, index_col):
    """Per-pixel slope of sif_z on delta_et, plus mean drought index."""
    sub = df_in.dropna(subset=['sif_z', 'delta_et', index_col]).copy()
    sub = sub[np.isfinite(sub['sif_z']) & np.isfinite(sub['delta_et'])
              & np.isfinite(sub[index_col])]

    out = []
    for (plat, plon), grp in sub.groupby(['lat', 'lon']):
        if len(grp) < MIN_OBS_PIXEL:
            continue
        x = grp['delta_et'].values
        y = grp['sif_z'].values
        if np.nanstd(x) == 0:
            continue
        slope, _, _, pval, _ = stats.linregress(x, y)
        out.append({'lat': plat, 'lon': plon, 'slope': slope, 'pval': pval,
                    'drought_mean': float(grp[index_col].mean()),
                    'n': len(grp)})
    return pd.DataFrame(out)


def _make_bivariate_map(df_in, index_col, index_label, fsuffix):
    tbl = _pixel_slope_table(df_in, index_col)
    if len(tbl) < 100:
        print('  {}: only {} pixels — skipped'.format(index_label, len(tbl)))
        return

    # Terciles. Drought axis is INVERTED (more negative index = drier = higher
    # tercile) so that "up" on the legend always means "more drought".
    tbl['x_ter'] = pd.qcut(tbl['slope'], 3, labels=False)
    tbl['y_ter'] = pd.qcut(-tbl['drought_mean'], 3, labels=False)

    _codes = np.full((n_lat, n_lon), -1, dtype=int)
    _lat_to_row_b = {round(float(v), 4): i for i, v in enumerate(CONUS_LAT)}
    _lon_to_col_b = {round(float(v), 4): i for i, v in enumerate(CONUS_LON)}
    for _, r in tbl.iterrows():
        ri = _lat_to_row_b.get(round(float(r['lat']), 4))
        ci = _lon_to_col_b.get(round(float(r['lon']), 4))
        if ri is not None and ci is not None:
            _codes[ri, ci] = int(r['y_ter']) * 3 + int(r['x_ter'])

    _codes = np.where(_use_final_mask, _codes, -1)

    # 9 discrete classes -> ListedColormap indexed by the class code.
    _flat = [BIV_GRID[j, i] for j in range(3) for i in range(3)]
    _cmap_biv = ListedColormap(_flat)
    _cmap_biv.set_bad('white', alpha=0)
    _norm_biv = BoundaryNorm(np.arange(-0.5, 9.5, 1.0), _cmap_biv.N)

    fig, ax = _map_fig(figsize=(9.5, 6.4))
    _albers_axes(ax)
    _add_basemap(ax)
    ax.pcolormesh(_X5070, _Y5070, np.ma.masked_where(_codes < 0, _codes),
                  cmap=_cmap_biv, norm=_norm_biv, shading='nearest',
                  rasterized=True, zorder=2)
    _draw_states(ax, lw=0.4, color='#333333', zorder=5)
    _draw_study_boundary(ax, lw=2.0, zorder=10)

    ax.text(0.015, 0.97, index_label, transform=ax.transAxes,
            fontsize=20, fontweight='bold', va='top', ha='left', zorder=15,
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=1.0,
                      edgecolor='none'))
    ax.text(0.99, 0.03, 'n={:,} pixels'.format(len(tbl)),
            transform=ax.transAxes, fontsize=13, ha='right', va='bottom',
            color='#555555')

    # Labels kept short: anything longer overruns the inset and the x- and
    # y-labels collide in the corner.
    _add_bivariate_legend(
        fig, BIV_GRID,
        'Buffering →',
        'Drought →',
    )

    fname = 'fig02b_bivariate_{}'.format(fsuffix)
    plt.savefig(str(figs / fname) + '.png', dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.savefig(str(figs / fname) + '.pdf', bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.show()
    print('  Saved:', fname + '.png / .pdf')


plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 22,
    'axes.labelsize': 19, 'axes.titlesize': 19,
    'xtick.labelsize': 16, 'ytick.labelsize': 16,
    'legend.fontsize': 14,
})

_BIV_INDICES = [
    ('spei30d',  'SPEI-30d',  'spei30d'),
    ('spei90d',  'SPEI-90d',  'spei90d'),
    ('spei180d', 'SPEI-180d', 'spei180d'),
    ('rzsm_z',   'RZSM z',    'rzsm_z'),
]

print()
for _col, _lbl, _suf in _BIV_INDICES:
    if _col not in df.columns or df[_col].notna().sum() == 0:
        print('  {}: column not available — skipped'.format(_lbl))
        continue
    _make_bivariate_map(df, _col, _lbl, _suf)

print()
print('Figure 2b complete.')


---

## 6. Figure B — SIF Response Across Irrigation Intensity and Drought Severity (Figure 3 Candidate)

Bins all pixel-month observations into 10 equal-count percentile groups based on HumanET value (decile 1 = lowest irrigation, decile 10 = highest). For each decile, computes the SIF z-score distribution split by drought severity.

**Drought severity categories:**
- No Drought: SPEI > −0.5
- Mild: −1.0 < SPEI ≤ −0.5
- Moderate: −1.5 < SPEI ≤ −1.0
- Severe: SPEI ≤ −1.5

**Option 1 (line plot):** x = HumanET decile, y = mean SIF z-score, one line per drought category. Directly answers: "as irrigation increases, does SIF recover more under drought?"

**Option 2 (violin plot):** x = drought severity category, half-violins colored by HumanET decile, showing the full distribution shift.

Both are exported; choose the clearest for the manuscript.

### 6.1 Compute HumanET Deciles and SIF Summaries


In [25]:
# ── Working dataset: complete cases ──────────────────────────────────────
_df_fig = df.dropna(subset=['sif_z', 'spei90d', 'delta_et']).copy()
_df_fig = _df_fig[np.isfinite(_df_fig['sif_z']) & np.isfinite(_df_fig['spei90d'])
                  & np.isfinite(_df_fig['delta_et'])].copy()

# ── HumanET decile bins (10 equal-count groups) ───────────────────────────
_df_fig['et_decile'] = pd.qcut(_df_fig['delta_et'], q=10, labels=False) + 1  # 1–10

# ── Drought severity categories ───────────────────────────────────────────
_drought_bins   = [-np.inf, -1.5, -1.0, -0.5, np.inf]
_drought_labels = ['Severe\n(SPEI<=-1.5)', 'Moderate\n(-1.5<SPEI<=-1.0)',
                   'Mild\n(-1.0<SPEI<=-0.5)', 'No Drought\n(SPEI>-0.5)']
_df_fig['drought_cat'] = pd.cut(
    _df_fig['spei90d'], bins=_drought_bins, labels=_drought_labels, right=True
)

print('HumanET decile bin boundaries (mm/month):')
_edges = _df_fig['delta_et'].quantile(np.linspace(0, 1, 11)).values
for i, (lo, hi) in enumerate(zip(_edges[:-1], _edges[1:])):
    n = (_df_fig['et_decile'] == i + 1).sum()
    print('  Decile {:2d}: {:6.1f} to {:6.1f}  (n={:,})'.format(i + 1, lo, hi, n))

print()
print('Drought category counts:')
print(_df_fig['drought_cat'].value_counts().to_string())


HumanET decile bin boundaries (mm/month):
  Decile  1: -133.4 to   -4.3  (n=41,973)
  Decile  2:   -4.3 to    3.3  (n=41,972)
  Decile  3:    3.3 to    8.8  (n=41,972)
  Decile  4:    8.8 to   13.7  (n=41,972)
  Decile  5:   13.7 to   18.4  (n=41,972)
  Decile  6:   18.4 to   23.5  (n=41,972)
  Decile  7:   23.5 to   29.3  (n=41,972)
  Decile  8:   29.3 to   37.2  (n=41,972)
  Decile  9:   37.2 to   51.0  (n=41,972)
  Decile 10:   51.0 to  233.8  (n=41,972)

Drought category counts:
drought_cat
No Drought\n(SPEI>-0.5)        304925
Mild\n(-1.0<SPEI<=-0.5)         64544
Moderate\n(-1.5<SPEI<=-1.0)     36776
Severe\n(SPEI<=-1.5)            13476


### 6.2 Figure B Option 1: Line Plot (Mean SIF by Decile × Drought Category)


In [26]:
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 22,
    'axes.labelsize': 19, 'axes.titlesize': 19,
    'xtick.labelsize': 18, 'ytick.labelsize': 18,
    'legend.fontsize': 12,
})

# ── Attach state labels so CA / Iowa case-study panels can be split out ──
if 'state' not in _df_fig.columns:
    _fig_px = _df_fig[['lat', 'lon']].drop_duplicates().reset_index(drop=True)
    _fig_px['state'] = _state_label_for_lonlat(_fig_px['lon'].values,
                                               _fig_px['lat'].values)
    _df_fig = _df_fig.merge(_fig_px, on=['lat', 'lon'], how='left')

print('Figure 3 subsets: CA {:,} obs | Iowa {:,} obs | all {:,} obs'.format(
    int((_df_fig['state'] == 'CA').sum()),
    int((_df_fig['state'] == 'IA').sum()),
    len(_df_fig),
))


def _fig03_lines(df_src, fname, title=None, n_bins=20):
    """Line plot of mean SIF z-score vs Human ET, one line per drought category.

    Legend sits inside the axes at lower right, sized small enough to clear
    the lines (which rise to the upper right).
    """
    _df_pos = df_src[df_src['delta_et'] >= 0].copy()
    if len(_df_pos) < 500:
        print('  {}: only {:,} obs — skipped'.format(fname, len(_df_pos)))
        return

    _nb = min(n_bins, max(4, len(_df_pos) // 200))
    _df_pos['et_bin'] = pd.qcut(_df_pos['delta_et'], q=_nb,
                                labels=False, duplicates='drop') + 1
    _bin_median_et = _df_pos.groupby('et_bin', observed=True)['delta_et'].median()

    _grp3 = _df_pos.groupby(['et_bin', 'drought_cat'], observed=True)
    _summary3 = _grp3['sif_z'].agg(['mean', 'sem', 'count']).reset_index()
    _summary3.columns = ['et_bin', 'drought_cat', 'mean_sifz', 'se_sifz', 'n']
    _summary3['et_mm'] = _summary3['et_bin'].map(_bin_median_et)

    fig, ax = plt.subplots(figsize=(9, 5.5))
    fig.patch.set_facecolor('white')

    _drought_order3 = list(reversed(_drought_labels))
    for cat in _drought_order3:
        _sub = _summary3[_summary3['drought_cat'] == cat].sort_values('et_mm')
        if len(_sub) == 0:
            continue
        col = DROUGHT_COLORS_LTC.get(cat, 'gray')
        ax.plot(_sub['et_mm'], _sub['mean_sifz'], color=col, linewidth=2.4,
                label=cat.replace('\n', ' '), zorder=5)
        ax.fill_between(_sub['et_mm'],
                        _sub['mean_sifz'] - _sub['se_sifz'],
                        _sub['mean_sifz'] + _sub['se_sifz'],
                        color=col, alpha=0.12, zorder=2)

    ax.axhline(0, color='#333333', linewidth=2.0, linestyle='-', zorder=3)
    ax.set_xlabel('Human ET [mm month⁻¹]', fontsize=19)
    ax.set_ylabel('Mean SIF z-score', fontsize=19)
    ax.set_xlim(left=0)
    ax.grid(alpha=0.25)

    if title:
        ax.set_title(title, fontsize=19, pad=8)

    # Rug marks: data density along the x-axis
    _rug_y = ax.get_ylim()[0]
    _rug = _df_pos['delta_et'].sample(n=min(3000, len(_df_pos)),
                                      random_state=42).values
    ax.plot(_rug, np.full_like(_rug, _rug_y), '|', color='#666666',
            markersize=4, markeredgewidth=0.5, alpha=0.18, zorder=1)
    # Reserve headroom at the bottom so the lower-right corner is empty
    # before the legend is placed. Without this the Severe-drought line runs
    # straight through the legend box in the California panel.
    _y0, _y1 = ax.get_ylim()
    ax.set_ylim(_y0 - 0.30 * (_y1 - _y0), _y1)

    # Legend inside the frame, lower right — compact so it clears the lines
    ax.legend(title='Drought severity', fontsize=11, title_fontsize=11,
              loc='lower right', framealpha=0.92, borderpad=0.5,
              labelspacing=0.35, handlelength=1.4, handletextpad=0.5)

    plt.tight_layout()
    plt.savefig(str(figs / fname) + '.png', dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.savefig(str(figs / fname) + '.pdf', bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.show()
    print('  Saved:', fname + '.png / .pdf')


# Full domain + the two case-study states
_fig03_lines(_df_fig, 'fig03_percentile_lines')
_fig03_lines(_df_fig[_df_fig['state'] == 'CA'],
             'fig03_percentile_lines_california', title='California')
_fig03_lines(_df_fig[_df_fig['state'] == 'IA'],
             'fig03_percentile_lines_iowa', title='Iowa')

print()
print('Figure 3 complete: full domain + California + Iowa.')


Figure 3 subsets: CA 12,349 obs | Iowa 46,893 obs | all 419,721 obs


  Saved: fig03_percentile_lines.png / .pdf


  Saved: fig03_percentile_lines_california.png / .pdf


  Saved: fig03_percentile_lines_iowa.png / .pdf

Figure 3 complete: full domain + California + Iowa.


### 6.3 Figure B Option 2: Violin Plot (SIF Distribution by Drought Category, Colored by ET Decile)

In [27]:
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 22,
    'axes.labelsize': 19, 'axes.titlesize': 19,
    'xtick.labelsize': 18, 'ytick.labelsize': 18,
    'legend.fontsize': 12,
})

# ── Helper: compute violin stats ──────────────────────────────────────────
def _violin_stats(data, positions, widths=0.4):
    """Compute kernel density estimate for a violin at a given position."""
    from scipy.stats import gaussian_kde
    results = []
    for pos, d in zip(positions, data):
        d = np.asarray(d)
        d = d[np.isfinite(d)]
        if len(d) < 5:
            results.append(None)
            continue
        kde      = gaussian_kde(d, bw_method='silverman')
        ymin     = float(np.percentile(d, 1))
        ymax     = float(np.percentile(d, 99))
        ys       = np.linspace(ymin, ymax, 200)
        kde_vals = kde(ys)
        kde_vals = kde_vals / kde_vals.max() * widths
        results.append({'pos': pos, 'ys': ys, 'kde': kde_vals,
                        'median': float(np.median(d)), 'n': len(d)})
    return results


# ── Only drought and no-drought groups; use 5 deciles to keep readable ────
_DECILE_SELECT = [1, 3, 5, 7, 10]
_CAT_ORDER = [
    'No Drought\n(SPEI>-0.5)',
    'Mild\n(-1.0<SPEI<=-0.5)',
    'Moderate\n(-1.5<SPEI<=-1.0)',
    'Severe\n(SPEI<=-1.5)',
]
_CAT_POSITIONS = [1, 2, 3, 4]
_CAT_SHORT     = ['No Drought', 'Mild', 'Moderate', 'Severe']

# Irrigation intensity colormap: light (low) → dark warm (high)
_irr_cmap  = plt.cm.viridis
_irr_norms = [0.05, 0.28, 0.52, 0.74, 0.95]

fig, ax = plt.subplots(figsize=(9, 5.5))
fig.patch.set_facecolor('white')

_violin_w     = 0.16
_decile_offsets = np.linspace(-0.35, 0.35, len(_DECILE_SELECT))

for di, (decile, offset, inorm) in enumerate(
        zip(_DECILE_SELECT, _decile_offsets, _irr_norms)):
    col      = _irr_cmap(inorm)
    col_dark = tuple(max(0, c - 0.15) for c in col[:3]) + (1.0,)

    for ci, (cat, cpos) in enumerate(zip(_CAT_ORDER, _CAT_POSITIONS)):
        _sub = _df_fig[(_df_fig['et_decile'] == decile) &
                       (_df_fig['drought_cat'] == cat)]['sif_z'].dropna().values
        if len(_sub) < 10:
            continue

        vstat = _violin_stats([_sub], [cpos + offset], widths=_violin_w * 0.9)
        if vstat[0] is None:
            continue
        vs = vstat[0]

        ax.fill_betweenx(vs['ys'],
                         (cpos + offset) - vs['kde'],
                         (cpos + offset) + vs['kde'],
                         color=col, alpha=0.8, linewidth=0)
        ax.plot((cpos + offset) - vs['kde'], vs['ys'],
                color=col_dark, linewidth=0.4, alpha=0.7)
        ax.plot((cpos + offset) + vs['kde'], vs['ys'],
                color=col_dark, linewidth=0.4, alpha=0.7)
        ax.hlines(vs['median'], (cpos + offset) - vs['kde'].max() * 0.7,
                  (cpos + offset) + vs['kde'].max() * 0.7,
                  color='#333333', linewidth=1.0, zorder=10)

ax.axhline(0, color='#666666', linewidth=0.8, linestyle='--', alpha=0.7, zorder=1)

_legend_patches = [
    Patch(facecolor=_irr_cmap(n), label='ET decile ' + str(d) +
          (' (low)' if d == 1 else ' (high)' if d == 10 else ''))
    for d, n in zip(_DECILE_SELECT, _irr_norms)
]
ax.legend(handles=_legend_patches, title='Human ET decile',
          loc='lower left', fontsize=7, title_fontsize=7, framealpha=0.9)

ax.set_xticks(_CAT_POSITIONS)
ax.set_xticklabels(_CAT_SHORT, fontsize=9)
ax.set_xlim(0.5, 4.5)
ax.set_xlabel('Drought severity category', fontsize=10)
ax.set_ylabel('SIF z-score  (0 = climatological mean)', fontsize=10)
ax.grid(axis='y', alpha=0.25)
ax.set_title(
    'SIF Distribution by Drought Severity and Irrigation Intensity\n'
    'CONUS Cropland Growing Season (April\u2013September, 2015\u20132024)',
    fontsize=11,
)

plt.tight_layout()
_fig03b_stem = figs / 'fig03_ridge_violin'
plt.savefig(str(_fig03b_stem) + '.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.savefig(str(_fig03b_stem) + '.pdf', bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print('Saved: fig03_ridge_violin.png / .pdf  (300 DPI)')


Saved: fig03_ridge_violin.png / .pdf  (300 DPI)


---

## 7. Multi-Index Figure 2 and Figure 3 Variants

Generates Figure 2 (4 drought-category slope maps) and Figure 3 (irrigation × drought
line plot) for each additional drought index:

| Index    | Column in parquet | Drought thresholds |
|----------|-------------------|--------------------|
| SPEI-30d | `spei30d`         | same as SPEI-90d   |
| SPEI-60d | `spei60d`         | same               |
| SPEI-90d | `spei90d`         | same (reference)   |
| SPEI-180d| `spei180d`        | same               |
| RZSM     | `rzsm_z`          | same (z-score)     |

**Prerequisites:** Run `05_download_spei_multiperiod.py`, `06_extract_nldas_soilm.py`,
and `07_extend_panel_parquet.py` before executing this section. If those haven't been run,
a guard cell will skip gracefully.

### 7.1 Check for new columns and reload parquet

In [28]:
# ── Check for extended parquet columns and set flags ──────────────────────
# Reload the parquet in case 07_extend_panel_parquet.py has been run since
# the notebook was opened.

_df_ext = pd.read_parquet(proc / 'regression' / 'df_combined_gs.parquet')
if 'date' not in _df_ext.columns:
    _df_ext['date'] = pd.to_datetime(_df_ext['yyyymm'], format='%Y%m')

# Drought index configuration
# key: (col_name, short_label, file_suffix, drought_thresholds)
_INDEX_CONFIG = [
    ('spei30d',  'SPEI-30d',  'spei30d',
     [-np.inf, -1.5, -1.0, -0.5, np.inf]),
    ('spei60d',  'SPEI-60d',  'spei60d',
     [-np.inf, -1.5, -1.0, -0.5, np.inf]),
    ('spei90d',  'SPEI-90d',  'spei90d',
     [-np.inf, -1.5, -1.0, -0.5, np.inf]),   # reference — already plotted above
    ('spei180d', 'SPEI-180d', 'spei180d',
     [-np.inf, -1.5, -1.0, -0.5, np.inf]),
    ('rzsm_z',   'RZSM-z',    'rzsm_z',
     [-np.inf, -1.5, -1.0, -0.5, np.inf]),    # standardized; same thresholds
]

_available = []
for col, label, fsuffix, thresholds in _INDEX_CONFIG:
    if col in _df_ext.columns and _df_ext[col].notna().sum() > 0:
        _available.append((col, label, fsuffix, thresholds))
        print('  AVAILABLE  : {} ({:,} non-null rows)'.format(
            label, _df_ext[col].notna().sum()))
    else:
        print('  NOT YET    : {} — run the download/extension scripts first'.format(label))

if not _available:
    print('\nNo extended drought index columns found.')
    print('Run 05_, 06_, 07_ scripts then re-run this cell.')
else:
    print('\n{} index(es) available for multi-index figures.'.format(len(_available)))

  AVAILABLE  : SPEI-30d (1,509,180 non-null rows)
  NOT YET    : SPEI-60d — run the download/extension scripts first
  AVAILABLE  : SPEI-90d (1,509,180 non-null rows)
  AVAILABLE  : SPEI-180d (513,900 non-null rows)
  AVAILABLE  : RZSM-z (830,580 non-null rows)

4 index(es) available for multi-index figures.


In [29]:
# ── Multi-index Figure 2: 4 drought-category slope maps per index ──────────
# Skips gracefully if index column not available.

if not _available:
    print('No data — skipping multi-index Figure 2.')
else:
    _drought_cat_labels = [
        ('No Drought\n(>{:.1f})'.format(-0.5),          'SPEI / RZSM-z > \u22120.5',    'no_drought'),
        ('Mild\n(\u22121.0 to \u22120.5)',               '\u22121.0 < idx \u2264 \u22120.5', 'mild'),
        ('Moderate\n(\u22121.5 to \u22121.0)',           '\u22121.5 < idx \u2264 \u22121.0', 'moderate'),
        ('Severe\n(\u2264\u22121.5)',                    'idx \u2264 \u22121.5',             'severe'),
    ]
    _drought_bins_ext   = [-np.inf, -1.5, -1.0, -0.5, np.inf]
    _cat_names_ext      = [d[0] for d in _drought_cat_labels]

    _lat_to_row_e = {round(float(v), 4): i for i, v in enumerate(CONUS_LAT)}
    _lon_to_col_e = {round(float(v), 4): i for i, v in enumerate(CONUS_LON)}

    for (idx_col, idx_label, idx_fsuffix, _) in _available:
        _df_idx = _df_ext.dropna(subset=['sif_z', idx_col, 'delta_et']).copy()
        _df_idx = _df_idx[
            np.isfinite(_df_idx['sif_z']) &
            np.isfinite(_df_idx[idx_col]) &
            np.isfinite(_df_idx['delta_et'])
        ].copy()

        _df_idx['drought_cat_ext'] = pd.cut(
            _df_idx[idx_col], bins=_drought_bins_ext,
            labels=_cat_names_ext, right=True,
        )

        # Per-category pixel regressions
        _slopes_this_idx = {}
        for cat_name, cat_title, cat_fsuffix in _drought_cat_labels:
            _df_cat = _df_idx[_df_idx['drought_cat_ext'] == cat_name]
            if len(_df_cat) < 50:
                _slopes_this_idx[cat_name] = pd.DataFrame()
                continue
            _ps = (_df_cat.groupby(['lat', 'lon'])
                   .apply(_pixel_slope_series).reset_index())
            _slopes_this_idx[cat_name] = _ps[_ps['pval'] < ALPHA_SPATIAL].copy()

        # Global symmetric vmax across all 4 categories
        _all_s = np.concatenate([
            _slopes_this_idx[c[0]]['slope'].dropna().values
            for c in _drought_cat_labels if len(_slopes_this_idx[c[0]]) > 0
        ])
        _sv_max_e = max(float(np.percentile(np.abs(_all_s), 97.5)), 0.005) if len(_all_s) else 0.01

        for cat_name, cat_title, cat_fsuffix in _drought_cat_labels:
            pix_cat_e = _slopes_this_idx[cat_name]
            _slope_grid_e = np.full((n_lat, n_lon), np.nan)
            for _, row in pix_cat_e.iterrows():
                ri = _lat_to_row_e.get(round(float(row['lat']), 4))
                ci = _lon_to_col_e.get(round(float(row['lon']), 4))
                if ri is not None and ci is not None:
                    _slope_grid_e[ri, ci] = row['slope']

            _slope_plot_e = np.where(_use_final_mask, _slope_grid_e, np.nan)

            fig, ax = plt.subplots(1, 1, figsize=(8, 5))
            fig.patch.set_facecolor('white')
            im = ax.pcolormesh(
                _X5070, _Y5070, _slope_plot_e,
                cmap=cmap_ploen_r_map, vmin=-_sv_max_e, vmax=_sv_max_e,
                shading='nearest', rasterized=True, zorder=2,
            )
            _albers_axes(ax)
            _add_basemap(ax)
            _draw_states(ax, lw=0.4, color='#333333', zorder=5)
            _draw_study_boundary(ax, lw=2.0, zorder=10)

            cb = fig.colorbar(im, ax=ax, orientation='horizontal',
                              fraction=0.04, pad=0.03, shrink=0.72)
            cb.set_label('Slope: SIF z-score per mm month\u207b\u00b9 HumanET', fontsize=15)
            cb.ax.tick_params(labelsize=14)
            cb.set_ticks([-_sv_max_e, 0, _sv_max_e])
            cb.set_ticklabels(['{:.3f}'.format(-_sv_max_e), '0', '{:.3f}'.format(_sv_max_e)])


            ax.text(0.015, 0.97, '{}\n{}'.format(idx_label, cat_title),
                    transform=ax.transAxes, fontsize=9, fontweight='bold',
                    va='top', ha='left',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.75))
            ax.text(0.99, 0.03, 'n={:,}'.format(len(pix_cat_e)),
                    transform=ax.transAxes, fontsize=7, ha='right', va='bottom',
                    color='#555555')

            fname_e = 'fig02_{}_slope_{}'.format(idx_fsuffix, cat_fsuffix)
            plt.tight_layout()
            plt.savefig(str(figs / fname_e) + '.png', dpi=300, bbox_inches='tight',
                        facecolor='white', edgecolor='none')
            plt.savefig(str(figs / fname_e) + '.pdf', bbox_inches='tight',
                        facecolor='white', edgecolor='none')
            plt.show()
            print('Saved:', fname_e)

    print('\nMulti-index Figure 2 complete.')


/tmp/ipykernel_2204395/1113935809.py:61: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(1, 1, figsize=(8, 5))


Saved: fig02_spei30d_slope_no_drought


Saved: fig02_spei30d_slope_mild


Saved: fig02_spei30d_slope_moderate


Saved: fig02_spei30d_slope_severe


...

Saved: fig02_spei180d_slope_severe


Saved: fig02_rzsm_z_slope_no_drought


Saved: fig02_rzsm_z_slope_mild


Saved: fig02_rzsm_z_slope_moderate


Saved: fig02_rzsm_z_slope_severe

Multi-index Figure 2 complete.


In [30]:
# ── Multi-index Figure 3: irrigation × drought line graph per index ────────
# Same design as the SPEI-90d version above: positive HumanET on x-axis,
# no markers, rug marks at bottom, ploen palette for drought categories.

if not _available:
    print('No data — skipping multi-index Figure 3.')
else:
    _drought_labels_ext = [
        'Severe\n(idx\u2264-1.5)',
        'Moderate\n(-1.5<idx\u2264-1.0)',
        'Mild\n(-1.0<idx\u2264-0.5)',
        'No Drought\n(idx>-0.5)',
    ]
    _drought_colors_ext = {
        _drought_labels_ext[3]: PALETTE_PLOEN[0],  # No Drought  → dark blue
        _drought_labels_ext[2]: PALETTE_PLOEN[1],  # Mild        → light blue
        _drought_labels_ext[1]: PALETTE_PLOEN[3],  # Moderate    → warm peach
        _drought_labels_ext[0]: PALETTE_PLOEN[4],  # Severe      → brownish red
    }

    for (idx_col, idx_label, idx_fsuffix, _) in _available:
        _df_i = _df_ext.dropna(subset=['sif_z', idx_col, 'delta_et']).copy()
        _df_i = _df_i[
            np.isfinite(_df_i['sif_z']) & np.isfinite(_df_i[idx_col])
            & np.isfinite(_df_i['delta_et'])
        ].copy()

        _df_i['drought_cat_f3'] = pd.cut(
            _df_i[idx_col],
            bins=[-np.inf, -1.5, -1.0, -0.5, np.inf],
            labels=_drought_labels_ext,
            right=True,
        )

        # Positive HumanET only (x-axis starts at 0)
        _df_pos_e = _df_i[_df_i['delta_et'] >= 0].copy()
        _df_pos_e['et_bin'] = pd.qcut(_df_pos_e['delta_et'], q=20, labels=False) + 1
        _bin_med = _df_pos_e.groupby('et_bin', observed=True)['delta_et'].median()

        _grp_e   = _df_pos_e.groupby(['et_bin', 'drought_cat_f3'], observed=True)
        _summ_e  = _grp_e['sif_z'].agg(['mean', 'sem', 'count']).reset_index()
        _summ_e.columns = ['et_bin', 'drought_cat', 'mean_sifz', 'se_sifz', 'n']
        _summ_e['et_mm'] = _summ_e['et_bin'].map(_bin_med)

        fig, ax = plt.subplots(figsize=(9, 5.5))
        fig.patch.set_facecolor('white')

        for cat in list(reversed(_drought_labels_ext)):
            _sub = _summ_e[_summ_e['drought_cat'] == cat].sort_values('et_mm')
            if len(_sub) == 0:
                continue
            col = _drought_colors_ext.get(cat, 'gray')
            ax.plot(_sub['et_mm'], _sub['mean_sifz'],
                    color=col, linewidth=2.2,
                    label=cat.replace('\n', ' ').replace('idx', idx_label),
                    zorder=5)
            ax.fill_between(_sub['et_mm'],
                            _sub['mean_sifz'] - _sub['se_sifz'],
                            _sub['mean_sifz'] + _sub['se_sifz'],
                            color=col, alpha=0.12, zorder=2)

        ax.axhline(0, color='#333333', linewidth=2.0, linestyle='-', zorder=3)
        ax.set_xlabel('Human ET [mm month\u207b\u00b9]', fontsize=19)
        ax.set_ylabel('Mean SIF z-score', fontsize=19)
        ax.set_xlim(left=0)
        ax.legend(title='Drought severity ({})'.format(idx_label),
                  fontsize=12, title_fontsize=12,
                  loc='upper left', bbox_to_anchor=(1.01, 1), framealpha=0.9)
        ax.grid(alpha=0.25)

        # Rug marks
        _rug_y_e = ax.get_ylim()[0]
        _rug_s   = _df_pos_e['delta_et'].sample(n=min(3000, len(_df_pos_e)),
                                                  random_state=42).values
        ax.plot(_rug_s, np.full_like(_rug_s, _rug_y_e),
                '|', color='#666666', markersize=4, markeredgewidth=0.5,
                alpha=0.18, zorder=1, clip_on=True)
        _y0e, _y1e = ax.get_ylim()
        ax.set_ylim(_y0e - 0.03 * (_y1e - _y0e), _y1e)

        fname_f3 = 'fig03_{}_percentile_lines'.format(idx_fsuffix)
        plt.tight_layout()
        plt.savefig(str(figs / fname_f3) + '.png', dpi=300, bbox_inches='tight',
                    facecolor='white', edgecolor='none')
        plt.savefig(str(figs / fname_f3) + '.pdf', bbox_inches='tight',
                    facecolor='white', edgecolor='none')
        plt.show()
        print('Saved:', fname_f3)

    print('\nMulti-index Figure 3 complete.')


Saved: fig03_spei30d_percentile_lines


Saved: fig03_spei90d_percentile_lines


Saved: fig03_spei180d_percentile_lines


Saved: fig03_rzsm_z_percentile_lines

Multi-index Figure 3 complete.


---

## 8. Controlling for Location: Fixed-Effects Regressions

The pooled OLS behind Figure 3 treats every pixel-month as an independent
observation. That leaves two confounds in place:

1. **Geographic bias.** Irrigated pixels differ from rainfed pixels in soil,
   crop mix, elevation and climate normals. A positive Human ET coefficient
   could reflect *where* irrigation happens rather than what irrigation *does*.
2. **Year shocks.** A single unusually wet or dry year shifts SPEI and SIF
   together across the whole domain.

A two-way fixed-effects panel addresses both. Pixel fixed effects absorb every
time-invariant property of a location, so identification comes only from
variation *within* a pixel over time — each pixel is compared against itself.
Year fixed effects absorb domain-wide annual shocks; month effects retain the
seasonal control from the original model. Standard errors are clustered by
pixel, which is also a partial answer to spatial autocorrelation: repeat
observations of the same pixel are no longer treated as independent.

Pixel effects are absorbed by within-transformation (demeaning each variable on
its pixel) rather than by ~7,000 dummy columns. The reported R-squared for the
FE model is therefore a **within** R-squared and is not directly comparable to
the pooled R-squared.


In [31]:
# ── Fixed-effects regressions controlling for location ───────────────────
import statsmodels.api as sm

_fe_df = df.dropna(subset=['sif_z', 'delta_et', 'spei90d']).copy()
_fe_df = _fe_df[np.isfinite(_fe_df['sif_z']) & np.isfinite(_fe_df['delta_et'])
                & np.isfinite(_fe_df['spei90d'])].copy()
_fe_df['spei_x_det'] = _fe_df['spei90d'] * _fe_df['delta_et']
_fe_df['pix'] = (_fe_df['lat'].round(4).astype(str) + '_'
                 + _fe_df['lon'].round(4).astype(str))

print('FE sample: {:,} obs across {:,} pixels'.format(
    len(_fe_df), _fe_df['pix'].nunique()))

_TERMS = ['spei90d', 'delta_et', 'spei_x_det']
_PRETTY = {'spei90d': 'SPEI-90d', 'delta_et': 'Human ET (dET)',
           'spei_x_det': 'SPEI x Human ET'}


def _run(y, X, label, cluster=None, note=''):
    res = (sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': cluster})
           if cluster is not None else sm.OLS(y, X).fit(cov_type='HC3'))
    print()
    print('=' * 68)
    print(label)
    print('  N = {:,}   R2 = {:.4f}   {}'.format(int(res.nobs), res.rsquared, note))
    print('-' * 68)
    for t in _TERMS:
        if t in res.params.index:
            b, se, p = res.params[t], res.bse[t], res.pvalues[t]
            stars = ('***' if p < .001 else '**' if p < .01
                     else '*' if p < .05 else '')
            print('  {:18s} b = {: .6f}   SE = {:.6f}   p = {:.3g} {}'.format(
                _PRETTY[t], b, se, p, stars))
    return res


# ── Model 1: pooled OLS (the Figure 3 specification) ─────────────────────
_m_d = pd.get_dummies(_fe_df['month'], prefix='m', drop_first=True).astype(float)
_X1 = sm.add_constant(pd.concat([_fe_df[_TERMS].astype(float), _m_d], axis=1))
_res_pooled = _run(_fe_df['sif_z'].astype(float), _X1,
                   'Model 1 — Pooled OLS, month FE, HC3 robust SE')

# ── Model 2: + year FE ───────────────────────────────────────────────────
_y_d = pd.get_dummies(_fe_df['year'], prefix='y', drop_first=True).astype(float)
_X2 = sm.add_constant(pd.concat([_fe_df[_TERMS].astype(float), _m_d, _y_d], axis=1))
_res_year = _run(_fe_df['sif_z'].astype(float), _X2,
                 'Model 2 — + year FE, clustered by pixel',
                 cluster=_fe_df['pix'].values)

# ── Model 3: pixel + year + month FE (within-transformed) ────────────────
_work = pd.concat([_fe_df[['sif_z'] + _TERMS].astype(float), _m_d, _y_d], axis=1)
_cols = list(_work.columns)
_work['pix'] = _fe_df['pix'].values
_demeaned = _work[_cols] - _work.groupby('pix', sort=False)[_cols].transform('mean')

_res_fe = _run(_demeaned['sif_z'],
               _demeaned[[c for c in _cols if c != 'sif_z']],
               'Model 3 — Pixel + year + month FE, clustered by pixel',
               cluster=_fe_df['pix'].values,
               note='(within R2; {:,} pixel effects absorbed)'.format(
                   _fe_df['pix'].nunique()))

# ── Marginal effect of irrigation across drought severity ────────────────
print()
print('=' * 68)
print('Marginal effect of +1 mm month-1 Human ET on SIF z-score')
print('  dSIF/dET = b_ET + b_interaction * SPEI')
print('-' * 68)
print('  {:<22s} {:>12s} {:>12s} {:>12s}'.format(
    'SPEI', 'Pooled', '+ year FE', 'Pixel+year FE'))
for _spei, _lbl in [(0.0, 'No drought (0)'), (-0.5, 'Mild (-0.5)'),
                    (-1.0, 'Moderate (-1.0)'), (-1.5, 'Severe (-1.5)'),
                    (-2.0, 'Extreme (-2.0)')]:
    _vals = [r.params['delta_et'] + r.params['spei_x_det'] * _spei
             for r in (_res_pooled, _res_year, _res_fe)]
    print('  {:<22s} {:>12.5f} {:>12.5f} {:>12.5f}'.format(_lbl, *_vals))

_b_et = _res_fe.params['delta_et']
_b_ix = _res_fe.params['spei_x_det']
_drop = 1 - (_b_et + _b_ix * -2.0) / _b_et
print()
print('Pixel+year FE: irrigation benefit at SPEI = -2.0 is {:.0f}% smaller'.format(
    100 * _drop))
print('than under no drought — the buffering-breakdown result, now net of')
print('every time-invariant difference between locations.')

# ── Export comparison table ──────────────────────────────────────────────
_rows_tab = []
for _lbl, _r in [('Pooled OLS', _res_pooled), ('+ Year FE', _res_year),
                 ('Pixel + Year FE', _res_fe)]:
    _row = {'Model': _lbl, 'N': int(_r.nobs), 'R2': round(_r.rsquared, 4)}
    for _t in _TERMS:
        _row[_PRETTY[_t]] = '{:.5f}'.format(_r.params[_t])
        _row[_PRETTY[_t] + ' SE'] = '{:.5f}'.format(_r.bse[_t])
    _rows_tab.append(_row)

_tab_fe = pd.DataFrame(_rows_tab)
_tab_fe.to_csv(figs / 'table_fixed_effects_comparison.csv', index=False)
print()
print('Saved: table_fixed_effects_comparison.csv')
_tab_fe


FE sample: 419,721 obs across 7,333 pixels



Model 1 — Pooled OLS, month FE, HC3 robust SE
  N = 419,721   R2 = 0.2082   
--------------------------------------------------------------------
  SPEI-90d           b =  0.165748   SE = 0.002123   p = 0 ***
  Human ET (dET)     b =  0.003410   SE = 0.000058   p = 0 ***
  SPEI x Human ET    b =  0.001060   SE = 0.000066   p = 9.06e-58 ***



Model 2 — + year FE, clustered by pixel
  N = 419,721   R2 = 0.2263   
--------------------------------------------------------------------
  SPEI-90d           b =  0.157338   SE = 0.003208   p = 0 ***
  Human ET (dET)     b =  0.003605   SE = 0.000090   p = 0 ***
  SPEI x Human ET    b =  0.001079   SE = 0.000099   p = 6.78e-28 ***



Model 3 — Pixel + year + month FE, clustered by pixel
  N = 419,721   R2 = 0.2319   (within R2; 7,333 pixel effects absorbed)
--------------------------------------------------------------------
  SPEI-90d           b =  0.172235   SE = 0.003310   p = 0 ***
  Human ET (dET)     b =  0.006041   SE = 0.000145   p = 0 ***
  SPEI x Human ET    b =  0.001274   SE = 0.000103   p = 2.53e-35 ***

Marginal effect of +1 mm month-1 Human ET on SIF z-score
  dSIF/dET = b_ET + b_interaction * SPEI
--------------------------------------------------------------------
  SPEI                         Pooled    + year FE Pixel+year FE
  No drought (0)              0.00341      0.00360      0.00604
  Mild (-0.5)                 0.00288      0.00307      0.00540
  Moderate (-1.0)             0.00235      0.00253      0.00477
  Severe (-1.5)               0.00182      0.00199      0.00413
  Extreme (-2.0)              0.00129      0.00145      0.00349

Pixel+year FE: irrigation benefit at SPEI = -2.0 is 42

,Model,N,R2,SPEI-90d,SPEI-90d SE,Human ET (dET),Human ET (dET) SE,SPEI x Human ET,SPEI x Human ET SE
0,Pooled OLS,419721,0.2082,0.16575,0.00212,0.00341,0.00006,0.00106,0.00007
1,+ Year FE,419721,0.2263,0.15734,0.00321,0.00360,0.00009,0.00108,0.00010
2,Pixel + Year FE,419721,0.2319,0.17223,0.00331,0.00604,0.00015,0.00127,0.00010
